# Discover, build, submit, and attach seqspec Documents for the WTC11-EC genome-wide and validation CRISPRi MACS screens

This notebook uses the confirmed guide libraries and the Sherlock paths
recovered from Dulguun's offboarding documentation:

```text
Genome-wide guide library:  IGVFFI9629WVUU
Validation guide library:   IGVFFI6745HXWN
```

It does **not** hard-code an unverified read layout. Instead, it:

1. discovers the raw FASTQs on Sherlock;
2. reconciles them with existing IGVF `SequenceFile` records;
3. groups SequenceFiles by screen and `file_set`;
4. downloads the confirmed Portal guide metadata and reads the `spacer`
   onlist;
5. samples real FASTQ reads and infers the guide offset and orientation by
   exact matching against the confirmed guide spacers;
6. checks that the inferred layout is reproducible across representative
   files;
7. writes one seqspec YAML per screen/file-set group;
8. validates every YAML with `seqspec format` and `seqspec check`;
9. posts each YAML as an IGVF `Document` with
   `document_type = library structure seqspec`;
10. PATCHes `seqspec_document` onto each matching SequenceFile;
11. writes final SequenceFile-to-Document reconciliation results.

Production POST/PATCH operations remain disabled until explicit confirmation
tokens are entered. The notebook stops rather than guessing whenever the
Portal crosswalk or read-layout inference is ambiguous.

## Version 2 — current IGVF SequenceFile discovery endpoints

The current IGVF API exposes the general search endpoint at
`/search` without a trailing slash. Version 1 called `/search/`,
which now returns HTTP 404.

This version uses `/search` first with the current dotted lab field
filter (`lab.@id`). It also includes two collection-endpoint
fallbacks through `/sequence-files/@@listing`, filters returned
records to the Jesse Engreitz lab in Python, and writes a diagnostic
TSV showing every attempted endpoint and HTTP response.

## Version 3 — IGVF-specific seqspec validation with local resource mirror

The previous validation used the default seqspec schema and treated
Portal resource reachability as part of the same check. That produced
three misleading classes of failures:

1. `sgrna_target` was rejected by the default schema even though it
   is allowed by the IGVF seqspec schema.
2. the guide onlist was reported missing by the generic remote-file
   checker;
3. all private/pending SequenceFile download URLs were reported
   missing because the checker uses an HTTP HEAD-style existence
   test that does not reliably handle authenticated IGVF download
   redirects.

Version 3 keeps the final submission YAML in Portal form, but also
builds a structurally identical local-resource validation YAML. It:

- validates the local mirror with `seqspec check -s igvf`;
- validates the submission YAML with
  `seqspec check -s igvf_onlist_skip`;
- independently probes representative Portal download URLs using
  authenticated ranged GET requests;
- accepts only the known remote-existence messages from the generic
  checker after the local and authenticated checks pass;
- treats every schema, geometry, region, primer, or unexpected file
  error as fatal.

## Version 4 — select an IGVF-capable seqspec executable

The Version 3 output showed that every authenticated Portal resource probe
passed and both checker error columns were empty, but both checker processes
returned exit code 2. This means the selected `seqspec` executable exited during
command-line argument parsing; validation never ran.

Version 4 probes every candidate executable before use. A candidate is accepted
only when `seqspec check --help` succeeds, the check command exposes the IGVF
skip modes, and the installed schema recognizes `sgrna_target`. The dedicated
`seqspec-0.4` environment is preferred over an older executable in the active
kernel's `PATH`.

## Version 5 — install the missing seqspec 0.4 environment correctly

The Version 4 audit is behaving correctly: the preferred
`seqspec-0.4` environment does not exist, while the installed legacy
environment contains seqspec 0.3.1 and lacks the required IGVF
validation modes and `sgrna_target` schema support.

Seqspec 0.4.0 requires Python 3.12 or newer. Version 5 corrects the
earlier Python 3.11 installation example and adds a gated bootstrap
cell that creates a separate Python 3.12 environment without changing
the existing seqspec 0.3.1 environment.

## Version 6 — pin a Sherlock-compatible NumPy wheel

The seqspec 0.4.0 installation reached its NumPy dependency, but
pip selected NumPy 2.5.3. Its Linux x86-64 wheel requires glibc
2.27/2.28, while Sherlock provides glibc 2.17. Pip therefore fell
back to the NumPy source distribution, whose local Meson build
failed while probing the system math library.

Version 6 writes a pip constraint that pins NumPy 2.2.6, which has
a CPython 3.12 manylinux2014 / glibc 2.17 wheel. The bootstrap then
installs all packages with `--only-binary=:all:` so pip must use
wheels and cannot silently launch another incompatible source build.

## Version 7 — isolate seqspec from the Jupyter Matplotlib backend

The seqspec 0.4.0 environment is installed correctly. The failure
occurred because the Jupyter kernel exported:

```text
MPLBACKEND=module://matplotlib_inline.backend_inline
```

into the external seqspec process. The dedicated seqspec environment
does not provide that notebook-only backend, so importing Matplotlib
failed before the seqspec command-line parser could start.

Version 7 gives every seqspec/install subprocess a clean headless
Matplotlib environment:

```text
MPLBACKEND=Agg
MPLCONFIGDIR=<writable workflow directory>
```

The current notebook's plotting backend is not modified. No package
reinstall is required.

## Version 8 — test IGVF skip modes functionally, not from help text

The installed seqspec 0.4.0 executable is working and exposes:

```text
-s SKIP, --skip SKIP
```

Its argparse help uses the explicit metavar `SKIP`, so the help text
does not enumerate the accepted values `igvf` and
`igvf_onlist_skip`. Version 7 incorrectly treated that omission as
evidence that the modes were unavailable.

Version 8 no longer searches the help text for mode names. It:

- requires the `--skip` option itself;
- directly probes `--skip igvf` and
  `--skip igvf_onlist_skip`;
- imports `seqspec.seqspec_check.filter_errors` in the candidate
  environment and verifies the actual filter behavior with synthetic
  diagnostics;
- retains the schema check for `sgrna_target`;
- records CLI and Python-level probe results in the executable audit.

## Version 9 — construct YAML attachments with an allowed MIME type

Version 8 reached Portal schema validation, but the attachment
generated by the `igvf_utils` `{"path": ...}` shortcut contained
`"type": null`. The external Python 3.9 MIME database did not infer
a type for `.yaml`, while the IGVF attachment schema requires an
allowed string MIME type.

Version 9 bypasses filename-based MIME guessing and constructs the
complete attachment object itself:

```text
download = original .yaml filename
type     = text/plain
href     = data:text/plain;base64,...
md5sum   = checksum of the YAML bytes
size     = YAML byte count
```

The object contains no `path` key, so `igvf_utils` leaves it
unchanged.

## Version 10 — attach a JSON serialization of the seqspec

Version 9 supplied valid YAML bytes with `type = text/plain`, but
the Portal rejected the `.yaml` download filename because its
extension did not match the declared MIME type.

The IGVF attachment schema permits `application/json`, and seqspec
0.4 reads JSON because JSON is valid YAML input. Version 10 therefore:

1. preserves each validated Portal-form YAML as the scientific source;
2. converts that YAML to an equivalent `.json` serialization;
3. re-validates the JSON serialization with the selected seqspec CLI;
4. attaches the `.json` file using
   `type = application/json`;
5. verifies that parsing YAML and JSON produces identical objects.

The SequenceFile still links to a `Document` whose
`document_type` is `library structure seqspec`; only the attachment
serialization changes to satisfy the Portal's extension–MIME check.

## Version 11 — JSON attachment and corrected production verification

Version 10 introduced the correct `application/json` seqspec
attachment but left one obsolete variable name in the post-submit
verification cell. Version 11 defines the expected `.json` attachment
filename directly from each final YAML stem and removes the obsolete
YAML-name variable.

The scientific and Portal behavior is otherwise unchanged:

- validated YAML remains the source representation;
- an object-equivalent JSON seqspec is attached;
- the Document retains
  `document_type = library structure seqspec`;
- production verification requires both the expected JSON filename
  and `application/json` MIME type.

## 1. Configuration

In [ ]:
from __future__ import annotations

import base64
import collections
import copy
import gzip
import hashlib
import json
import os
import re
import shutil
import subprocess
import time
from getpass import getpass
from pathlib import Path
from typing import Any, Iterable, Sequence
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
import yaml

OAK_ROOT = Path(
    os.environ.get(
        "OAK",
        "/oak/stanford/groups/engreitz",
    )
)

OUTPUT_ROOT = (
    OAK_ROOT
    / "Users/tri/tmp/"
    "igvf_crispri_macs_seqspec"
)
WORK_DIR = OUTPUT_ROOT / "_work"
YAML_DIR = OUTPUT_ROOT / "seqspec_yaml"
LOG_DIR = OUTPUT_ROOT / "logs"
ROW_DIR = WORK_DIR / "one_row_inputs"
GUIDE_DIR = WORK_DIR / "guide_metadata"

for directory in (
    OUTPUT_ROOT,
    WORK_DIR,
    YAML_DIR,
    LOG_DIR,
    ROW_DIR,
    GUIDE_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

AWARD = "/awards/HG011972/"
LAB = "/labs/jesse-engreitz/"

GUIDE_LIBRARY_ACCESSIONS = {
    "genome_wide": "IGVFFI9629WVUU",
    "validation": "IGVFFI6745HXWN",
}

# Optional local overrides. Leave blank to download the Portal file.
GUIDE_LIBRARY_LOCAL_OVERRIDES = {
    "genome_wide": "",
    "validation": "",
}

SCREEN_CONFIG = {
    "genome_wide": {
        "display_name": (
            "WTC11-EC genome-wide CRISPRi "
            "MACS screen"
        ),
        "raw_roots": [
            OAK_ROOT
            / "Users/munger/Full_EC_Screen/"
            "230421_EC_Screen/fastq",
            OAK_ROOT
            / "Users/munger/Full_EC_Screen/"
            "23508_Second_EC_Run_Demux/fastq",
            OAK_ROOT
            / "Users/munger/Full_EC_Screen/"
            "230531_EC_Screen_Second_Seq/fastq",
        ],
        "fastq_path_manifest": (
            OAK_ROOT
            / "Users/munger/Full_EC_Screen/"
            "230421_EC_Screen/fastq_file_paths.txt"
        ),
        "sequence_date": "17 April 2023",
        "description": (
            "Guide-amplicon sequencing for the "
            "WTC11-EC genome-wide CRISPRi "
            "magnetic-separation differentiation screen."
        ),
        "portal_terms": [
            "genome",
            "ec_screen",
            "ec-screen",
            "230417",
            "230505",
            "full_ec_screen",
        ],
    },
    "validation": {
        "display_name": (
            "WTC11-EC focused validation "
            "CRISPRi MACS screen"
        ),
        "raw_roots": [
            OAK_ROOT
            / "Users/chloemc/"
            "GenomeWideValidation/"
            "CRISPRiFlowFish",
        ],
        "fastq_path_manifest": None,
        "sequence_date": "20 October 2023",
        "description": (
            "Guide-amplicon sequencing for the "
            "focused WTC11-EC CRISPRi validation "
            "screen of genome-wide candidates."
        ),
        "portal_terms": [
            "validation",
            "genomewidevalidation",
            "crispriflowfish",
            "231020",
        ],
    },
}

# The methods describe a custom Read 1 sequencing primer and a custom
# Index 1 sequencing primer. The FASTQ inference below determines where
# the confirmed guide spacer appears after the Read 1 primer.
CUSTOM_READ1_PRIMER = (
    "CGATTTCTTGGCTTTATATATCTTGTG"
    "GAAAGGACGAAACACCG"
)
CUSTOM_INDEX1_PRIMER = (
    "AAGGCTAGTCCGTTATCAACTTGAAAAA"
    "GTGGCACCG"
)

SEQSPEC_VERSION = "0.3.0"
LIBRARY_PROTOCOL = (
    "CRISPR screen (NTR:0000520)"
)
LIBRARY_KIT = "Custom"
SEQUENCE_PROTOCOL = (
    "Illumina NextSeq 550 (EFO:0008566)"
)
SEQUENCE_KIT = (
    "NextSeq 500/550 High-Output v2.5 Kit"
)

# Read-layout inference controls.
FASTQ_RECORDS_PER_FILE = 50_000
REPRESENTATIVE_FILES_PER_GROUP = 3
MIN_GUIDE_MATCH_FRACTION = 0.50
MIN_BEST_VS_SECOND_GAP = 0.05
MIN_MODAL_LENGTH_FRACTION = 0.90
FIXED_BASE_CONSENSUS_FRACTION = 0.95

# Optional manual SequenceFile mappings:
# {"/absolute/path/to/file.fastq.gz": "IGVFFI..."}
SEQUENCE_FILE_ACCESSION_OVERRIDES: dict[str, str] = {}

# Optional explicit seqspec executable.
SEQSPEC_EXECUTABLE_OVERRIDE = ""

PREFERRED_SEQSPEC_ENV = (
    OAK_ROOT
    / "Users/tri/envs/seqspec-0.4"
)
LEGACY_SEQSPEC_ENV = (
    OAK_ROOT
    / "Users/tri/envs/seqspec"
)


# Explicit installation gate for the dedicated current environment.
INSTALL_SEQSPEC_0_4_NOW = False
MAMBA_EXECUTABLE_OVERRIDE = ""

# NumPy 2.5.x wheels require a newer glibc than Sherlock provides.
# NumPy 2.2.6 has a CPython 3.12 manylinux2014 (glibc 2.17) wheel.
SEQSPEC_NUMPY_PIN = "2.2.6"

IGVF_PYTHON = Path(
    "/oak/stanford/groups/engreitz/Users/tri/"
    "envs/igvf_utils/bin/python"
)
IU_REGISTER = Path(
    "/oak/stanford/groups/engreitz/Users/tri/"
    "envs/igvf_utils/igvf_utils/igvf_utils/"
    "MetaDataRegistration/iu_register.py"
)

# Production gates. Leave blank until dry runs have passed.
DOCUMENT_PRODUCTION_CONFIRMATION = ""
SEQUENCE_PATCH_PRODUCTION_CONFIRMATION = ""

print("Output root:", OUTPUT_ROOT)
print("GW guide library:", GUIDE_LIBRARY_ACCESSIONS["genome_wide"])
print("Validation guide library:", GUIDE_LIBRARY_ACCESSIONS["validation"])
print("Custom Read 1 primer length:", len(CUSTOM_READ1_PRIMER))
print("Custom Index 1 primer length:", len(CUSTOM_INDEX1_PRIMER))

## 2. Utility functions

Portal-linked fields may be returned as accessions, paths, UUIDs, embedded
dictionaries, or lists. The helpers below normalize those representations
before comparison.

In [ ]:

def seqspec_subprocess_env() -> dict[str, str]:
    """Return a clean, headless environment for external seqspec commands."""
    environment = os.environ.copy()

    # Jupyter commonly exports a notebook-only module backend. That value
    # must not leak into a separate Python environment that may not contain
    # matplotlib-inline.
    environment["MPLBACKEND"] = "Agg"

    # Keep Matplotlib cache/config writes out of $HOME and in a workflow-
    # local, writable directory.
    matplotlib_config_dir = (
        WORK_DIR / "matplotlib_config_seqspec"
    )
    matplotlib_config_dir.mkdir(
        parents=True,
        exist_ok=True,
    )
    environment["MPLCONFIGDIR"] = str(
        matplotlib_config_dir
    )

    return environment


def write_tsv(
    frame: pd.DataFrame,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    frame.to_csv(
        path,
        sep="\t",
        index=False,
        lineterminator="\n",
    )


def md5sum(
    path: Path,
    block_size: int = 16 * 1024 * 1024,
) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(block_size),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def sha256sum(
    path: Path,
    block_size: int = 16 * 1024 * 1024,
) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(block_size),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def safe_stem(value: object) -> str:
    return re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(value),
    ).strip("_")


def split_link_items(
    value: object,
) -> list[object]:
    if value is None:
        return []
    if isinstance(value, str):
        return [
            item.strip()
            for item in value.split(",")
            if item.strip()
        ]
    if isinstance(
        value,
        (list, tuple, set),
    ):
        return list(value)
    return [value]


def terminal_identifier(
    value: object,
) -> str:
    if isinstance(value, dict):
        value = (
            value.get("accession")
            or value.get("@id")
            or value.get("uuid")
            or ""
        )
    text = str(value or "").strip().strip("/")
    return (
        text.rsplit("/", 1)[-1]
        if text
        else ""
    )


def direct_link_tokens(
    value: object,
) -> set[str]:
    tokens: set[str] = set()

    for item in split_link_items(value):
        if isinstance(item, dict):
            for key in (
                "accession",
                "uuid",
                "@id",
            ):
                token = terminal_identifier(
                    item.get(key, "")
                )
                if token:
                    tokens.add(token)

            for alias in split_link_items(
                item.get("aliases", [])
            ):
                alias_text = str(
                    alias
                ).strip()
                if alias_text:
                    tokens.add(alias_text)
        else:
            token = terminal_identifier(
                item
            )
            if token:
                tokens.add(token)

            text = str(item).strip()
            if (
                ":" in text
                and not text.startswith("/")
            ):
                tokens.add(text)

    return tokens


def first_accession(
    value: object,
    prefix: str,
) -> str:
    candidates = sorted(
        token
        for token in direct_link_tokens(value)
        if token.startswith(prefix)
    )
    return candidates[0] if candidates else ""


def normalize_path_text(
    value: object,
) -> str:
    text = str(value or "").strip()
    text = text.replace(
        "/Users/dulguun/oak",
        str(OAK_ROOT),
    )
    text = text.replace(
        "/oak/stanford/groups/engreitz/User/",
        "/oak/stanford/groups/engreitz/Users/",
    )
    return text


def row_without_empty_values(
    row: dict,
) -> dict:
    return {
        key: value
        for key, value in row.items()
        if str(value).strip() != ""
    }


def reverse_complement(
    sequence: str,
) -> str:
    table = str.maketrans(
        "ACGTNacgtn",
        "TGCANtgcan",
    )
    return sequence.translate(table)[::-1]


def portal_urls(
    identifier: str,
) -> list[str]:
    token = str(identifier).strip().strip("/")
    base = (
        "https://api.data.igvf.org/"
        f"{token}/"
    )
    return [
        (
            base
            + "?format=json&datastore=database"
        ),
        (
            base
            + "?format=json&datastore=database"
            "&frame=object"
        ),
        (
            base
            + "?format=json&datastore=database"
            "&frame=edit"
        ),
    ]


def portal_auth() -> tuple[str, str] | None:
    if (
        os.environ.get("IGVF_API_KEY")
        and os.environ.get(
            "IGVF_SECRET_KEY"
        )
    ):
        return (
            os.environ["IGVF_API_KEY"],
            os.environ["IGVF_SECRET_KEY"],
        )
    return None


def portal_get(
    identifier: str,
    *,
    timeout: int = 60,
) -> dict | None:
    errors: list[str] = []

    for url in portal_urls(identifier):
        response = requests.get(
            url,
            auth=portal_auth(),
            headers={
                "Accept": "application/json"
            },
            timeout=timeout,
        )

        if response.status_code == 404:
            continue

        if response.ok:
            return response.json()

        errors.append(
            f"{response.status_code} {url}: "
            f"{response.text[:500]}"
        )

    if errors:
        raise RuntimeError(
            "Portal GET failed:\n"
            + "\n".join(errors)
        )
    return None


def get_with_retry(
    identifier: str,
    *,
    attempts: int = 5,
) -> dict | None:
    last_error: Exception | None = None

    for attempt in range(
        1,
        attempts + 1,
    ):
        try:
            record = portal_get(identifier)
            if record is not None:
                return record
        except Exception as error:
            last_error = error

        if attempt < attempts:
            time.sleep(
                min(
                    2 ** (attempt - 1),
                    10,
                )
            )

    if last_error is not None:
        raise last_error
    return None


_LINK_IDENTITY_CACHE: dict[
    str,
    set[str],
] = {}


def resolved_link_tokens(
    value: object,
) -> set[str]:
    direct = direct_link_tokens(value)
    resolved = set(direct)

    for token in list(direct):
        if token in _LINK_IDENTITY_CACHE:
            resolved.update(
                _LINK_IDENTITY_CACHE[token]
            )
            continue

        identities = {token}
        try:
            record = portal_get(token)
        except Exception:
            record = None

        if record:
            identities.update(
                direct_link_tokens(record)
            )

        _LINK_IDENTITY_CACHE[token] = (
            identities
        )
        resolved.update(identities)

    return resolved


def links_equal(
    left: object,
    right: object,
) -> bool:
    left_tokens = resolved_link_tokens(
        left
    )
    right_tokens = resolved_link_tokens(
        right
    )
    return bool(
        left_tokens
        and right_tokens
        and left_tokens & right_tokens
    )


def portal_record_id(
    record: dict | None,
) -> str:
    record = record or {}
    return str(
        record.get("@id")
        or record.get("accession")
        or record.get("uuid")
        or ""
    )


def portal_record_url(
    record: dict | None,
) -> str:
    record = record or {}
    portal_id = str(
        record.get("@id") or ""
    )
    return (
        "https://data.igvf.org"
        + portal_id
        if portal_id
        else ""
    )


def enter_igvf_credentials() -> None:
    api_key = getpass(
        "IGVF API key: "
    ).strip()
    secret_key = getpass(
        "IGVF secret key: "
    ).strip()

    if not api_key or not secret_key:
        raise ValueError(
            "Both IGVF credentials are required"
        )

    os.environ[
        "IGVF_API_KEY"
    ] = api_key
    os.environ[
        "IGVF_SECRET_KEY"
    ] = secret_key

    del api_key, secret_key

    print(
        "IGVF credentials are set for "
        "the current notebook kernel."
    )


def show_credential_status() -> None:
    for variable in (
        "IGVF_API_KEY",
        "IGVF_SECRET_KEY",
    ):
        value = os.environ.get(
            variable,
            "",
        )
        print(
            f"{variable}: "
            f"{'SET' if value else 'MISSING'}, "
            f"length={len(value)}"
        )


def run_iu_register(
    *,
    profile: str,
    input_path: Path,
    dry_run: bool,
    patch: bool,
    label: str,
) -> dict:
    if not IGVF_PYTHON.is_file():
        raise FileNotFoundError(
            IGVF_PYTHON
        )
    if not IU_REGISTER.is_file():
        raise FileNotFoundError(
            IU_REGISTER
        )
    if not input_path.is_file():
        raise FileNotFoundError(
            input_path
        )
    if portal_auth() is None:
        raise RuntimeError(
            "IGVF credentials are not set"
        )

    timestamp = time.strftime(
        "%Y%m%d_%H%M%S"
    )
    mode = (
        "dryrun"
        if dry_run
        else "submit"
    )
    log_path = (
        LOG_DIR
        / (
            f"{safe_stem(label)}_"
            f"{mode}_{timestamp}.log"
        )
    )

    command = [
        str(IGVF_PYTHON),
        str(IU_REGISTER),
        "-m",
        "prod",
        "-p",
        profile,
        "--tries",
        "3",
        "--delay",
        "10",
        "--backoff",
        "2",
    ]

    if dry_run:
        command.append("-d")
    if patch:
        command.append("--patch")

    command.extend(
        ["-i", str(input_path)]
    )

    print(
        "\nCommand:",
        " ".join(command),
    )
    print("Log:", log_path)

    completed = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
        check=False,
    )

    log_path.write_text(
        completed.stdout,
        encoding="utf-8",
    )
    print(completed.stdout)

    fatal = re.search(
        (
            r"Traceback|ValidationFailure|"
            r"Failed to POST|Failed to PATCH|"
            r"\b(?:401|403|409|422) "
            r"Client Error\b"
        ),
        completed.stdout,
        flags=re.IGNORECASE,
    )

    success = (
        completed.returncode == 0
        and fatal is None
    )

    if not success:
        raise RuntimeError(
            "igvf_utils failed; review "
            f"{log_path}"
        )

    return {
        "success": True,
        "log_path": str(log_path),
    }


print("Utility functions loaded.")

## 3. Enter IGVF credentials

Credentials are needed to retrieve editable Portal metadata and later to
POST/PATCH records. Rerun this section after any kernel restart.

In [ ]:
enter_igvf_credentials()
show_credential_status()

## 4. Retrieve the confirmed guide libraries

The source files are the exact Portal guide libraries confirmed for guide
counting. The notebook requires a non-empty `spacer` column and reports the
actual spacer-length distribution rather than assuming a conventional
20-nt guide length.

In [ ]:
def download_candidate_urls(
    record: dict,
) -> list[str]:
    accession = str(
        record.get("accession") or ""
    )
    portal_id = str(
        record.get("@id") or ""
    )

    candidates: list[str] = []

    for key in (
        "href",
        "download_url",
    ):
        value = record.get(key)
        if value:
            candidates.append(str(value))

    if portal_id and accession:
        base = (
            "https://api.data.igvf.org"
            + portal_id
            + "@@download/"
        )
        for suffix in (
            ".tsv.gz",
            ".tsv",
            ".txt.gz",
            ".txt",
        ):
            candidates.append(
                base
                + accession
                + suffix
            )

    # Stable order without duplicates.
    output: list[str] = []
    seen: set[str] = set()
    for candidate in candidates:
        if candidate not in seen:
            output.append(candidate)
            seen.add(candidate)

    return output


def download_portal_file(
    record: dict,
    output_path: Path,
) -> tuple[Path, str]:
    errors: list[str] = []

    for url in download_candidate_urls(
        record
    ):
        try:
            response = requests.get(
                url,
                auth=portal_auth(),
                timeout=180,
            )
        except Exception as error:
            errors.append(
                f"{url}: {type(error).__name__}: "
                f"{error}"
            )
            continue

        if not response.ok:
            errors.append(
                f"{url}: HTTP "
                f"{response.status_code}"
            )
            continue

        output_path.write_bytes(
            response.content
        )

        expected_md5 = str(
            record.get("md5sum") or ""
        )
        if (
            expected_md5
            and md5sum(output_path)
            != expected_md5
        ):
            output_path.unlink(
                missing_ok=True
            )
            errors.append(
                f"{url}: MD5 mismatch"
            )
            continue

        return output_path, url

    raise RuntimeError(
        "Could not download Portal file:\n"
        + "\n".join(errors)
    )


def read_maybe_gzip_table(
    path: Path,
) -> pd.DataFrame:
    with path.open("rb") as handle:
        magic = handle.read(2)

    compression = (
        "gzip"
        if magic == b"\x1f\x8b"
        else None
    )

    return pd.read_csv(
        path,
        sep="\t",
        compression=compression,
        dtype=str,
        keep_default_na=False,
        na_values=[],
        low_memory=False,
    )


guide_library_rows: list[dict] = []
guide_metadata_frames: dict[
    str,
    pd.DataFrame,
] = {}
guide_spacer_sets: dict[
    str,
    set[str],
] = {}
guide_portal_records: dict[
    str,
    dict,
] = {}
guide_download_urls: dict[
    str,
    str,
] = {}

for screen, accession in (
    GUIDE_LIBRARY_ACCESSIONS.items()
):
    record = get_with_retry(accession)
    if record is None:
        raise RuntimeError(
            "Guide library not found on Portal: "
            f"{accession}"
        )

    guide_portal_records[screen] = (
        record
    )

    override = str(
        GUIDE_LIBRARY_LOCAL_OVERRIDES[
            screen
        ]
    ).strip()

    if override:
        local_path = Path(override)
        if not local_path.is_file():
            raise FileNotFoundError(
                local_path
            )
        source_url = "local override"
    else:
        local_path = (
            GUIDE_DIR
            / f"{accession}.download"
        )
        local_path, source_url = (
            download_portal_file(
                record,
                local_path,
            )
        )

    frame = read_maybe_gzip_table(
        local_path
    )

    if "spacer" not in frame.columns:
        raise RuntimeError(
            f"{accession} has no `spacer` "
            "column"
        )

    spacers = (
        frame["spacer"]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    spacers = spacers.loc[
        spacers.str.fullmatch(
            r"[ACGT]+"
        )
    ]

    if spacers.empty:
        raise RuntimeError(
            f"No valid spacer sequences in "
            f"{accession}"
        )

    length_counts = (
        spacers.str.len()
        .value_counts()
        .sort_index()
        .to_dict()
    )

    if len(length_counts) != 1:
        raise RuntimeError(
            f"{accession} has multiple "
            "spacer lengths: "
            f"{length_counts}"
        )

    guide_metadata_frames[screen] = (
        frame
    )
    guide_spacer_sets[screen] = set(
        spacers
    )
    guide_download_urls[screen] = (
        source_url
    )

    guide_library_rows.append(
        {
            "screen": screen,
            "accession": accession,
            "portal_id": str(
                record.get("@id") or ""
            ),
            "portal_md5sum": str(
                record.get("md5sum") or ""
            ),
            "portal_file_size": record.get(
                "file_size",
                "",
            ),
            "local_path": str(local_path),
            "download_url": source_url,
            "rows": len(frame),
            "valid_spacers": len(
                spacers
            ),
            "unique_spacers": len(
                set(spacers)
            ),
            "spacer_length_counts": (
                json.dumps(
                    length_counts,
                    sort_keys=True,
                )
            ),
        }
    )

guide_library_inventory = (
    pd.DataFrame(guide_library_rows)
)
GUIDE_LIBRARY_INVENTORY_PATH = (
    WORK_DIR
    / "guide_library_inventory.tsv"
)
write_tsv(
    guide_library_inventory,
    GUIDE_LIBRARY_INVENTORY_PATH,
)

print(
    guide_library_inventory.to_string(
        index=False
    )
)
print(
    "\nSaved:",
    GUIDE_LIBRARY_INVENTORY_PATH,
)

## 5. Discover local FASTQs on Sherlock

The canonical paths below come from the offboarding document. The third GW
root is treated as a lower-priority mirror because the documentation notes
that it may duplicate the second run.

In [ ]:
def find_fastqs_under(
    root: Path,
) -> list[Path]:
    if not root.is_dir():
        return []

    command = [
        "find",
        str(root),
        "-type",
        "f",
        "(",
        "-name",
        "*.fastq.gz",
        "-o",
        "-name",
        "*.fq.gz",
        "-o",
        "-name",
        "*.fastq",
        "-o",
        "-name",
        "*.fq",
        ")",
        "-print",
    ]

    completed = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            f"find failed under {root}:\n"
            f"{completed.stderr}"
        )

    return [
        Path(line.strip())
        for line in (
            completed.stdout.splitlines()
        )
        if line.strip()
    ]


def paths_from_manifest(
    path: Path,
) -> list[Path]:
    if not path.is_file():
        return []

    found: list[Path] = []
    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    for token in re.split(
        r"[\t,\r\n ]+",
        text,
    ):
        token = normalize_path_text(
            token
        )
        if re.search(
            r"\.(?:fastq|fq)(?:\.gz)?$",
            token,
            flags=re.IGNORECASE,
        ):
            found.append(Path(token))

    return found


fastq_rows: list[dict] = []

for screen, config in (
    SCREEN_CONFIG.items()
):
    candidate_paths: list[
        tuple[Path, int, str]
    ] = []

    manifest_path = config.get(
        "fastq_path_manifest"
    )
    if manifest_path:
        for path in paths_from_manifest(
            Path(manifest_path)
        ):
            candidate_paths.append(
                (
                    path,
                    -1,
                    "fastq path manifest",
                )
            )

    for priority, root in enumerate(
        config["raw_roots"]
    ):
        for path in find_fastqs_under(
            Path(root)
        ):
            candidate_paths.append(
                (
                    path,
                    priority,
                    str(root),
                )
            )

    seen: set[str] = set()
    for path, priority, source in (
        candidate_paths
    ):
        normalized = str(
            path.expanduser()
        )
        if normalized in seen:
            continue
        seen.add(normalized)

        exists = path.is_file()

        fastq_rows.append(
            {
                "screen": screen,
                "path": normalized,
                "basename": path.name,
                "root_priority": priority,
                "discovery_source": source,
                "exists": exists,
                "size_bytes": (
                    path.stat().st_size
                    if exists
                    else 0
                ),
            }
        )

local_fastqs = pd.DataFrame(
    fastq_rows
)

if local_fastqs.empty:
    raise RuntimeError(
        "No FASTQ candidates were discovered"
    )

LOCAL_FASTQ_DISCOVERY_PATH = (
    WORK_DIR
    / "local_fastq_discovery.tsv"
)
write_tsv(
    local_fastqs,
    LOCAL_FASTQ_DISCOVERY_PATH,
)

print(
    local_fastqs.groupby(
        ["screen", "exists"]
    ).size()
)
print(
    "\nSaved:",
    LOCAL_FASTQ_DISCOVERY_PATH,
)

## 6. Retrieve Jesse Engreitz lab SequenceFiles and build a local–Portal crosswalk

Matching priority is:

1. user-supplied accession override;
2. exact normalized `submitted_file_name`;
3. unique basename match;
4. MD5 match for otherwise unresolved local files.

Only Portal records that reconcile to one of the documented raw-data roots
are included in the seqspec workflow.

In [ ]:
def extract_portal_records(
    payload: object,
) -> list[dict]:
    """Extract a record list from search or collection responses."""
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        records = None
        for key in (
            "@graph",
            "results",
            "items",
        ):
            candidate = payload.get(key)
            if isinstance(candidate, list):
                records = candidate
                break
        if records is None:
            records = []
    else:
        records = []

    return [
        record
        for record in records
        if isinstance(record, dict)
    ]


def sequence_file_lab_matches(
    record: dict,
) -> bool:
    """Match embedded/path/string lab representations safely."""
    expected_lab_token = terminal_identifier(LAB)
    lab_tokens = direct_link_tokens(
        record.get("lab", "")
    )
    return expected_lab_token in lab_tokens


def portal_search_sequence_files() -> list[dict]:
    """
    Retrieve Jesse Engreitz SequenceFiles using current IGVF endpoints.

    The official API currently defines the general endpoint as `/search`
    (without a trailing slash) and the collection endpoint as
    `/sequence-files/@@listing`.
    """
    attempts = [
        {
            "name": "current search endpoint with dotted lab filter",
            "url": "https://api.data.igvf.org/search",
            "params": {
                "type": "SequenceFile",
                "lab.@id": LAB,
                "limit": "all",
                "frame": "object",
                "format": "json",
            },
            "server_filtered_by_lab": True,
        },
        {
            "name": "SequenceFile collection with lab_id filter",
            "url": (
                "https://api.data.igvf.org/"
                "sequence-files/@@listing"
            ),
            "params": {
                "lab_id": LAB,
                "limit": "all",
                "frame": "object",
                "format": "json",
            },
            "server_filtered_by_lab": True,
        },
        {
            "name": "unfiltered SequenceFile collection fallback",
            "url": (
                "https://api.data.igvf.org/"
                "sequence-files/@@listing"
            ),
            "params": {
                "limit": "all",
                "frame": "object",
                "format": "json",
            },
            "server_filtered_by_lab": False,
        },
    ]

    diagnostic_rows: list[dict] = []
    selected_records: list[dict] | None = None
    selected_attempt = ""

    for attempt in attempts:
        try:
            response = requests.get(
                attempt["url"],
                params=attempt["params"],
                auth=portal_auth(),
                headers={
                    "Accept": "application/json"
                },
                timeout=180,
                allow_redirects=True,
            )
        except Exception as error:
            diagnostic_rows.append(
                {
                    "attempt": attempt["name"],
                    "requested_url": attempt["url"],
                    "final_url": "",
                    "params": json.dumps(
                        attempt["params"],
                        sort_keys=True,
                    ),
                    "status_code": "request_error",
                    "records_returned": 0,
                    "lab_records_returned": 0,
                    "response_preview": (
                        f"{type(error).__name__}: {error}"
                    ),
                    "selected": False,
                }
            )
            continue

        response_preview = response.text[:500].replace(
            "\n",
            " ",
        )

        if not response.ok:
            diagnostic_rows.append(
                {
                    "attempt": attempt["name"],
                    "requested_url": attempt["url"],
                    "final_url": response.url,
                    "params": json.dumps(
                        attempt["params"],
                        sort_keys=True,
                    ),
                    "status_code": response.status_code,
                    "records_returned": 0,
                    "lab_records_returned": 0,
                    "response_preview": response_preview,
                    "selected": False,
                }
            )
            continue

        try:
            payload = response.json()
        except Exception as error:
            diagnostic_rows.append(
                {
                    "attempt": attempt["name"],
                    "requested_url": attempt["url"],
                    "final_url": response.url,
                    "params": json.dumps(
                        attempt["params"],
                        sort_keys=True,
                    ),
                    "status_code": response.status_code,
                    "records_returned": 0,
                    "lab_records_returned": 0,
                    "response_preview": (
                        "Invalid JSON response: "
                        f"{type(error).__name__}: {error}; "
                        f"body={response_preview}"
                    ),
                    "selected": False,
                }
            )
            continue

        records = extract_portal_records(payload)
        lab_records = [
            record
            for record in records
            if sequence_file_lab_matches(record)
        ]

        # Some filtered list frames can omit the embedded lab field. When
        # the request was explicitly lab-filtered, accept all returned
        # records only when every returned record lacks a lab value.
        lab_fields_absent = bool(records) and all(
            not record.get("lab")
            for record in records
        )
        if (
            not lab_records
            and attempt["server_filtered_by_lab"]
            and lab_fields_absent
        ):
            lab_records = records

        selected = bool(lab_records)
        diagnostic_rows.append(
            {
                "attempt": attempt["name"],
                "requested_url": attempt["url"],
                "final_url": response.url,
                "params": json.dumps(
                    attempt["params"],
                    sort_keys=True,
                ),
                "status_code": response.status_code,
                "records_returned": len(records),
                "lab_records_returned": len(lab_records),
                "response_preview": response_preview,
                "selected": selected,
            }
        )

        if selected:
            selected_records = lab_records
            selected_attempt = attempt["name"]
            break

    diagnostic = pd.DataFrame(diagnostic_rows)
    PORTAL_SEQUENCEFILE_QUERY_DIAGNOSTIC_PATH = (
        WORK_DIR
        / "portal_sequencefile_query_diagnostic.tsv"
    )
    write_tsv(
        diagnostic,
        PORTAL_SEQUENCEFILE_QUERY_DIAGNOSTIC_PATH,
    )

    print("SequenceFile API query diagnostics:")
    print(diagnostic.to_string(index=False))
    print(
        "\nSaved:",
        PORTAL_SEQUENCEFILE_QUERY_DIAGNOSTIC_PATH,
    )

    if selected_records is None:
        raise RuntimeError(
            "No current IGVF SequenceFile endpoint returned readable "
            "Jesse Engreitz lab records. Review "
            f"{PORTAL_SEQUENCEFILE_QUERY_DIAGNOSTIC_PATH}"
        )

    print(
        f"\nSelected endpoint: {selected_attempt}; "
        f"records={len(selected_records):,}"
    )
    return selected_records

sequence_records = (
    portal_search_sequence_files()
)

portal_rows: list[dict] = []

for record in sequence_records:
    accession = str(
        record.get("accession") or ""
    )
    submitted_name = normalize_path_text(
        record.get(
            "submitted_file_name",
            "",
        )
    )
    alias_text = ",".join(
        map(
            str,
            record.get(
                "aliases",
                [],
            ),
        )
    )

    file_set = first_accession(
        record.get("file_set", ""),
        "IGVF",
    )
    if not file_set:
        file_set = terminal_identifier(
            record.get(
                "file_set",
                "",
            )
        )

    portal_rows.append(
        {
            "accession": accession,
            "aliases": alias_text,
            "submitted_file_name": (
                submitted_name
            ),
            "basename": (
                Path(
                    submitted_name
                ).name
                if submitted_name
                else ""
            ),
            "md5sum": str(
                record.get("md5sum") or ""
            ),
            "file_size": record.get(
                "file_size",
                "",
            ),
            "file_set": file_set,
            "illumina_read_type": str(
                record.get(
                    "illumina_read_type",
                    "",
                )
            ),
            "read_names": ",".join(
                map(
                    str,
                    record.get(
                        "read_names",
                        [],
                    ),
                )
            ),
            "sequencing_run": str(
                record.get(
                    "sequencing_run",
                    "",
                )
            ),
            "lane": str(
                record.get("lane", "")
            ),
            "minimum_read_length": (
                record.get(
                    "minimum_read_length",
                    "",
                )
            ),
            "maximum_read_length": (
                record.get(
                    "maximum_read_length",
                    "",
                )
            ),
            "sequencing_platform": (
                terminal_identifier(
                    record.get(
                        "sequencing_platform",
                        "",
                    )
                )
            ),
            "sequencing_kit": str(
                record.get(
                    "sequencing_kit",
                    "",
                )
            ),
            "portal_id": str(
                record.get("@id") or ""
            ),
            "portal_url": (
                portal_record_url(record)
            ),
        }
    )

portal_sequence_files = pd.DataFrame(
    portal_rows
)
PORTAL_SEQUENCE_FILES_PATH = (
    WORK_DIR
    / "portal_sequence_files_all.tsv"
)
write_tsv(
    portal_sequence_files,
    PORTAL_SEQUENCE_FILES_PATH,
)

portal_by_accession = {
    row["accession"]: row
    for row in portal_rows
    if row["accession"]
}

exact_path_index: dict[
    str,
    list[dict],
] = collections.defaultdict(list)
basename_index: dict[
    str,
    list[dict],
] = collections.defaultdict(list)
md5_index: dict[
    str,
    list[dict],
] = collections.defaultdict(list)
size_index: dict[
    int,
    list[dict],
] = collections.defaultdict(list)

for row in portal_rows:
    submitted_name = str(
        row["submitted_file_name"]
    ).strip()
    if submitted_name:
        exact_path_index[
            submitted_name
        ].append(row)
    if row["basename"]:
        basename_index[
            row["basename"]
        ].append(row)
    if row["md5sum"]:
        md5_index[
            row["md5sum"]
        ].append(row)
    try:
        size_value = int(
            row["file_size"]
        )
    except (
        TypeError,
        ValueError,
    ):
        size_value = 0
    if size_value > 0:
        size_index[
            size_value
        ].append(row)

crosswalk_rows: list[dict] = []

for local in local_fastqs.to_dict(
    "records"
):
    path_text = str(local["path"])
    path = Path(path_text)
    override_accession = (
        SEQUENCE_FILE_ACCESSION_OVERRIDES.get(
            path_text,
            "",
        )
    )

    candidates: list[dict] = []
    match_method = ""
    local_md5 = ""

    if override_accession:
        candidate = portal_by_accession.get(
            override_accession
        )
        if candidate is None:
            raise RuntimeError(
                "SequenceFile override was "
                "not found in Portal search: "
                f"{override_accession}"
            )
        candidates = [candidate]
        match_method = "manual override"
    else:
        candidates = exact_path_index.get(
            normalize_path_text(
                path_text
            ),
            [],
        )
        if len(candidates) == 1:
            match_method = (
                "exact submitted_file_name"
            )
        elif len(candidates) != 1:
            candidates = basename_index.get(
                path.name,
                [],
            )
            if len(candidates) == 1:
                match_method = (
                    "unique basename"
                )

    # Hash a large FASTQ only when at least one Portal file has the
    # same byte size. This avoids hashing every unsubmitted/intermediate
    # FASTQ discovered under a broad recursive root.
    if (
        len(candidates) != 1
        and local["exists"]
    ):
        same_size_candidates = (
            size_index.get(
                int(local["size_bytes"]),
                [],
            )
        )
        if same_size_candidates:
            local_md5 = md5sum(path)
            candidates = [
                candidate
                for candidate in (
                    same_size_candidates
                )
                if candidate["md5sum"]
                == local_md5
            ]
            if len(candidates) == 1:
                match_method = (
                    "file-size plus MD5"
                )

    candidate = (
        candidates[0]
        if len(candidates) == 1
        else None
    )

    crosswalk_rows.append(
        {
            **local,
            "local_md5sum": local_md5,
            "candidate_count": len(
                candidates
            ),
            "match_method": match_method,
            "accession": (
                candidate["accession"]
                if candidate
                else ""
            ),
            "aliases": (
                candidate["aliases"]
                if candidate
                else ""
            ),
            "portal_submitted_file_name": (
                candidate[
                    "submitted_file_name"
                ]
                if candidate
                else ""
            ),
            "portal_md5sum": (
                candidate["md5sum"]
                if candidate
                else ""
            ),
            "file_size": (
                candidate["file_size"]
                if candidate
                else ""
            ),
            "file_set": (
                candidate["file_set"]
                if candidate
                else ""
            ),
            "illumina_read_type": (
                candidate[
                    "illumina_read_type"
                ]
                if candidate
                else ""
            ),
            "read_names": (
                candidate["read_names"]
                if candidate
                else ""
            ),
            "sequencing_run": (
                candidate[
                    "sequencing_run"
                ]
                if candidate
                else ""
            ),
            "lane": (
                candidate["lane"]
                if candidate
                else ""
            ),
            "minimum_read_length": (
                candidate[
                    "minimum_read_length"
                ]
                if candidate
                else ""
            ),
            "maximum_read_length": (
                candidate[
                    "maximum_read_length"
                ]
                if candidate
                else ""
            ),
            "portal_id": (
                candidate["portal_id"]
                if candidate
                else ""
            ),
            "portal_url": (
                candidate["portal_url"]
                if candidate
                else ""
            ),
            "matched": candidate is not None,
        }
    )

sequence_crosswalk = pd.DataFrame(
    crosswalk_rows
)

# When the same uploaded FASTQ exists in multiple local mirrors, retain
# the highest-priority path for each SequenceFile accession.
matched = sequence_crosswalk.loc[
    sequence_crosswalk["matched"]
].copy()

matched = matched.sort_values(
    [
        "screen",
        "accession",
        "root_priority",
        "path",
    ]
)

matched["canonical_local_copy"] = (
    ~matched.duplicated(
        subset=["accession"],
        keep="first",
    )
)

sequence_crosswalk = (
    sequence_crosswalk.merge(
        matched[
            [
                "path",
                "accession",
                "canonical_local_copy",
            ]
        ],
        on=[
            "path",
            "accession",
        ],
        how="left",
    )
)
sequence_crosswalk[
    "canonical_local_copy"
] = (
    sequence_crosswalk[
        "canonical_local_copy"
    ]
    .fillna(False)
    .astype(bool)
)

SEQUENCE_CROSSWALK_PATH = (
    WORK_DIR
    / "sequencefile_local_portal_crosswalk.tsv"
)
write_tsv(
    sequence_crosswalk,
    SEQUENCE_CROSSWALK_PATH,
)

canonical_sequence_files = (
    sequence_crosswalk.loc[
        sequence_crosswalk[
            "canonical_local_copy"
        ]
    ].copy()
)

print(
    canonical_sequence_files[
        [
            "screen",
            "path",
            "accession",
            "file_set",
            "illumina_read_type",
            "match_method",
        ]
    ].to_string(index=False)
)
print(
    "\nSaved:",
    SEQUENCE_CROSSWALK_PATH,
)

if canonical_sequence_files.empty:
    raise RuntimeError(
        "No documented local FASTQ was "
        "matched to a Portal SequenceFile"
    )

if canonical_sequence_files[
    "file_set"
].astype(str).eq("").any():
    raise RuntimeError(
        "At least one matched SequenceFile "
        "has no resolved file_set"
    )

## 7. Normalize read roles and define seqspec groups

The default supported case is a demultiplexed, single-end guide-amplicon
FASTQ set. The notebook stops if a file set contains multiple read roles
because that requires a larger library model rather than an implicit guess.

In [ ]:
def infer_read_role(
    row: pd.Series,
) -> str:
    explicit = str(
        row.get(
            "illumina_read_type",
            "",
        )
    ).strip().upper()

    explicit_map = {
        "R1": "R1",
        "READ 1": "R1",
        "READ1": "R1",
        "R2": "R2",
        "READ 2": "R2",
        "READ2": "R2",
        "I1": "I1",
        "INDEX 1": "I1",
        "INDEX1": "I1",
        "I2": "I2",
        "INDEX 2": "I2",
        "INDEX2": "I2",
    }
    if explicit in explicit_map:
        return explicit_map[explicit]

    text = " ".join(
        [
            str(row.get("basename", "")),
            str(row.get("aliases", "")),
            str(row.get("read_names", "")),
        ]
    ).upper()

    patterns = {
        "R1": (
            r"(?:^|[_\-.])R1(?:[_\-.]|$)|"
            r"READ[_ -]?1"
        ),
        "R2": (
            r"(?:^|[_\-.])R2(?:[_\-.]|$)|"
            r"READ[_ -]?2"
        ),
        "I1": (
            r"(?:^|[_\-.])I1(?:[_\-.]|$)|"
            r"INDEX[_ -]?1"
        ),
        "I2": (
            r"(?:^|[_\-.])I2(?:[_\-.]|$)|"
            r"INDEX[_ -]?2"
        ),
    }

    matches = [
        role
        for role, pattern in (
            patterns.items()
        )
        if re.search(pattern, text)
    ]

    if len(matches) == 1:
        return matches[0]

    # These historical guide-screen outputs were generated as
    # demultiplexed single-end guide reads. An otherwise unlabelled
    # single read is provisionally designated R1 and then validated
    # against the confirmed guide onlist below.
    return "R1_UNLABELLED"


canonical_sequence_files[
    "read_role"
] = canonical_sequence_files.apply(
    infer_read_role,
    axis=1,
)

group_summary = (
    canonical_sequence_files.groupby(
        ["screen", "file_set"],
        dropna=False,
    )
    .agg(
        sequence_files=(
            "accession",
            "nunique",
        ),
        read_roles=(
            "read_role",
            lambda values: ",".join(
                sorted(set(values))
            ),
        ),
        local_files=(
            "path",
            "nunique",
        ),
    )
    .reset_index()
)

GROUP_SUMMARY_PATH = (
    WORK_DIR
    / "seqspec_group_summary.tsv"
)
write_tsv(
    group_summary,
    GROUP_SUMMARY_PATH,
)

print(
    group_summary.to_string(
        index=False
    )
)

unsupported = group_summary.loc[
    group_summary[
        "read_roles"
    ].str.contains(",")
    | ~group_summary[
        "read_roles"
    ].isin(
        {"R1", "R1_UNLABELLED"}
    )
]

if not unsupported.empty:
    raise RuntimeError(
        "At least one file set contains "
        "multiple or unsupported read roles. "
        f"Review {GROUP_SUMMARY_PATH}"
    )

## 8. Infer guide layout from real FASTQ reads

Exact spacer matching is evaluated at every possible offset on both the
forward read and its reverse complement. The best result must exceed both
an absolute match threshold and a margin over the second-best layout.

A 19-nt guide library can therefore support a 20-nt sequenced guide region
when the FASTQ contains a fixed leading G; the inferred prefix is represented
explicitly rather than silently changing the onlist.

In [ ]:
def open_fastq_text(
    path: Path,
):
    with path.open("rb") as raw:
        magic = raw.read(2)

    if magic == b"\x1f\x8b":
        return gzip.open(
            path,
            "rt",
            encoding="ascii",
            errors="replace",
        )

    return path.open(
        "rt",
        encoding="ascii",
        errors="replace",
    )


def sample_fastq_sequences(
    path: Path,
    *,
    maximum_records: int,
) -> list[str]:
    sequences: list[str] = []

    with open_fastq_text(path) as handle:
        while len(sequences) < maximum_records:
            header = handle.readline()
            if not header:
                break
            sequence = handle.readline()
            plus = handle.readline()
            quality = handle.readline()

            if not (
                sequence
                and plus
                and quality
            ):
                raise RuntimeError(
                    f"Truncated FASTQ record in {path}"
                )

            sequence = (
                sequence.strip().upper()
            )
            if not re.fullmatch(
                r"[ACGTN]+",
                sequence,
            ):
                raise RuntimeError(
                    "Unexpected FASTQ sequence "
                    f"characters in {path}"
                )

            sequences.append(sequence)

    if not sequences:
        raise RuntimeError(
            f"No FASTQ records read from {path}"
        )

    return sequences


def modal_read_length(
    sequences: Sequence[str],
) -> tuple[int, float, dict[int, int]]:
    counts = collections.Counter(
        map(len, sequences)
    )
    modal_length, modal_count = (
        counts.most_common(1)[0]
    )
    fraction = (
        modal_count
        / len(sequences)
    )
    return (
        int(modal_length),
        float(fraction),
        dict(sorted(counts.items())),
    )


def score_spacer_layouts(
    sequences: Sequence[str],
    spacers: set[str],
) -> pd.DataFrame:
    guide_length = len(
        next(iter(spacers))
    )
    minimum_length = min(
        map(len, sequences)
    )

    rows: list[dict] = []

    for orientation in (
        "forward",
        "reverse_complement",
    ):
        oriented = (
            list(sequences)
            if orientation == "forward"
            else [
                reverse_complement(sequence)
                for sequence in sequences
            ]
        )

        for offset in range(
            0,
            minimum_length
            - guide_length
            + 1,
        ):
            matched = sum(
                sequence[
                    offset : offset
                    + guide_length
                ]
                in spacers
                for sequence in oriented
            )

            rows.append(
                {
                    "orientation": orientation,
                    "offset": offset,
                    "guide_length": (
                        guide_length
                    ),
                    "matched_reads": matched,
                    "total_reads": len(
                        oriented
                    ),
                    "match_fraction": (
                        matched
                        / len(oriented)
                    ),
                }
            )

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "match_fraction",
                "matched_reads",
                "offset",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


def infer_consensus_segments(
    sequences: Sequence[str],
    *,
    target_offset: int,
    target_length: int,
    modal_length: int,
) -> dict:
    modal_sequences = [
        sequence
        for sequence in sequences
        if len(sequence) == modal_length
    ]

    if not modal_sequences:
        raise RuntimeError(
            "No sequences at modal read length"
        )

    matrix = np.array(
        [
            list(sequence)
            for sequence in modal_sequences
        ],
        dtype="U1",
    )

    consensus_chars: list[str] = []
    consensus_support: list[float] = []

    for position in range(modal_length):
        counts = collections.Counter(
            matrix[:, position]
        )
        base, count = counts.most_common(1)[0]
        consensus_chars.append(base)
        consensus_support.append(
            count / len(modal_sequences)
        )

    target_end = (
        target_offset
        + target_length
    )

    def segment_runs(
        start: int,
        end: int,
        label: str,
    ) -> list[dict]:
        if start >= end:
            return []

        output: list[dict] = []
        run_start = start
        run_is_fixed = (
            consensus_support[start]
            >= FIXED_BASE_CONSENSUS_FRACTION
        )

        for position in range(
            start + 1,
            end,
        ):
            is_fixed = (
                consensus_support[position]
                >= FIXED_BASE_CONSENSUS_FRACTION
            )
            if is_fixed != run_is_fixed:
                output.append(
                    {
                        "start": run_start,
                        "end": position,
                        "kind": (
                            "fixed"
                            if run_is_fixed
                            else "random"
                        ),
                        "sequence": (
                            "".join(
                                consensus_chars[
                                    run_start:position
                                ]
                            )
                            if run_is_fixed
                            else "X"
                            * (
                                position
                                - run_start
                            )
                        ),
                        "minimum_support": (
                            min(
                                consensus_support[
                                    run_start:position
                                ]
                            )
                        ),
                        "label": label,
                    }
                )
                run_start = position
                run_is_fixed = is_fixed

        output.append(
            {
                "start": run_start,
                "end": end,
                "kind": (
                    "fixed"
                    if run_is_fixed
                    else "random"
                ),
                "sequence": (
                    "".join(
                        consensus_chars[
                            run_start:end
                        ]
                    )
                    if run_is_fixed
                    else "X"
                    * (end - run_start)
                ),
                "minimum_support": min(
                    consensus_support[
                        run_start:end
                    ]
                ),
                "label": label,
            }
        )

        return output

    return {
        "modal_sequence_count": len(
            modal_sequences
        ),
        "consensus": "".join(
            consensus_chars
        ),
        "consensus_support": (
            consensus_support
        ),
        "prefix_segments": segment_runs(
            0,
            target_offset,
            "read_prefix",
        ),
        "suffix_segments": segment_runs(
            target_end,
            modal_length,
            "read_suffix",
        ),
    }


inference_rows: list[dict] = []
layout_score_frames: list[pd.DataFrame] = []
group_layouts: dict[
    tuple[str, str],
    dict,
] = {}

for (
    screen,
    file_set,
), group in (
    canonical_sequence_files.groupby(
        ["screen", "file_set"]
    )
):
    group = group.sort_values(
        ["accession", "path"]
    )
    representatives = group.head(
        REPRESENTATIVE_FILES_PER_GROUP
    )

    spacers = guide_spacer_sets[
        screen
    ]
    per_file_layouts: list[dict] = []

    for row in representatives.to_dict(
        "records"
    ):
        path = Path(row["path"])
        sequences = sample_fastq_sequences(
            path,
            maximum_records=(
                FASTQ_RECORDS_PER_FILE
            ),
        )

        modal_length, modal_fraction, length_counts = (
            modal_read_length(sequences)
        )

        if (
            modal_fraction
            < MIN_MODAL_LENGTH_FRACTION
        ):
            raise RuntimeError(
                f"Read lengths are too heterogeneous in {path}: "
                f"{length_counts}"
            )

        modal_sequences = [
            sequence
            for sequence in sequences
            if len(sequence) == modal_length
        ]

        scores = score_spacer_layouts(
            modal_sequences,
            spacers,
        )
        best = scores.iloc[0]
        second = scores.iloc[1]

        scores = scores.head(25).copy()
        scores.insert(
            0,
            "screen",
            screen,
        )
        scores.insert(
            1,
            "file_set",
            file_set,
        )
        scores.insert(
            2,
            "accession",
            row["accession"],
        )
        scores.insert(
            3,
            "path",
            row["path"],
        )
        layout_score_frames.append(
            scores
        )

        best_fraction = float(
            best["match_fraction"]
        )
        score_gap = (
            best_fraction
            - float(
                second[
                    "match_fraction"
                ]
            )
        )

        if (
            best_fraction
            < MIN_GUIDE_MATCH_FRACTION
        ):
            raise RuntimeError(
                "Guide matching is below the "
                "required threshold for "
                f"{path}: {best_fraction:.3f}"
            )
        if (
            score_gap
            < MIN_BEST_VS_SECOND_GAP
        ):
            raise RuntimeError(
                "Guide-layout inference is "
                "ambiguous for "
                f"{path}: gap={score_gap:.3f}"
            )

        orientation = str(
            best["orientation"]
        )
        oriented_sequences = (
            modal_sequences
            if orientation == "forward"
            else [
                reverse_complement(
                    sequence
                )
                for sequence in (
                    modal_sequences
                )
            ]
        )

        offset = int(best["offset"])
        guide_length = int(
            best["guide_length"]
        )

        matched_sequences = [
            sequence
            for sequence in (
                oriented_sequences
            )
            if sequence[
                offset : offset
                + guide_length
            ]
            in spacers
        ]

        consensus = infer_consensus_segments(
            matched_sequences,
            target_offset=offset,
            target_length=guide_length,
            modal_length=modal_length,
        )

        layout = {
            "screen": screen,
            "file_set": file_set,
            "accession": (
                row["accession"]
            ),
            "path": row["path"],
            "read_role": row["read_role"],
            "orientation": orientation,
            "offset": offset,
            "guide_length": guide_length,
            "match_fraction": best_fraction,
            "score_gap": score_gap,
            "modal_read_length": (
                modal_length
            ),
            "modal_length_fraction": (
                modal_fraction
            ),
            "length_counts": (
                length_counts
            ),
            **consensus,
        }
        per_file_layouts.append(layout)

        inference_rows.append(
            {
                key: (
                    json.dumps(
                        value,
                        sort_keys=True,
                    )
                    if isinstance(
                        value,
                        (
                            dict,
                            list,
                        ),
                    )
                    else value
                )
                for key, value in (
                    layout.items()
                )
                if key
                not in {
                    "consensus_support"
                }
            }
        )

    signatures = {
        (
            layout["orientation"],
            layout["offset"],
            layout["guide_length"],
            layout["modal_read_length"],
        )
        for layout in per_file_layouts
    }

    if len(signatures) != 1:
        raise RuntimeError(
            "Representative FASTQs disagree "
            "on guide layout for "
            f"{screen}/{file_set}: "
            f"{signatures}"
        )

    if {
        layout["read_role"]
        for layout in per_file_layouts
    } - {"R1", "R1_UNLABELLED"}:
        raise RuntimeError(
            "The guide-bearing read is not "
            f"R1 for {screen}/{file_set}"
        )

    if {
        layout["orientation"]
        for layout in per_file_layouts
    } != {"forward"}:
        raise RuntimeError(
            "Guide reads require reverse-"
            "orientation modeling for "
            f"{screen}/{file_set}; manual "
            "review is required"
        )

    # Use the first representative after confirming identical structural
    # signatures. Consensus support is still recorded per file.
    group_layouts[
        (screen, file_set)
    ] = per_file_layouts[0]

layout_inference = pd.DataFrame(
    inference_rows
)
LAYOUT_INFERENCE_PATH = (
    WORK_DIR
    / "fastq_guide_layout_inference.tsv"
)
write_tsv(
    layout_inference,
    LAYOUT_INFERENCE_PATH,
)

layout_scores = pd.concat(
    layout_score_frames,
    ignore_index=True,
)
LAYOUT_SCORES_PATH = (
    WORK_DIR
    / "fastq_guide_layout_scores_top25.tsv"
)
write_tsv(
    layout_scores,
    LAYOUT_SCORES_PATH,
)

print(
    layout_inference[
        [
            "screen",
            "file_set",
            "accession",
            "orientation",
            "offset",
            "guide_length",
            "match_fraction",
            "modal_read_length",
        ]
    ].to_string(index=False)
)
print(
    "\nSaved:",
    LAYOUT_INFERENCE_PATH,
)

## 9. Build submission and local-validation seqspec YAMLs

For every screen/file-set group the notebook writes two specifications from
the same inferred assay structure:

- **submission YAML** — Portal accessions, stable IGVF download URLs, and
  `urltype: https`;
- **local validation YAML** — the same region/read structure, but with the
  already reconciled Sherlock FASTQs and downloaded guide metadata represented
  as `urltype: local`.

The guide region retains `region_type: sgrna_target`, which belongs to the
IGVF-specific seqspec schema. The local mirror exists only for validation and
is never submitted as the Document attachment.

In [ ]:
class SeqspecDumper(
    yaml.SafeDumper
):
    pass


class Assay(dict):
    pass


class Read(dict):
    pass


class File(dict):
    pass


class Region(dict):
    pass


class Onlist(dict):
    pass


def represent_tagged(
    tag: str,
):
    def representer(
        dumper,
        data,
    ):
        return dumper.represent_mapping(
            tag,
            data,
        )

    return representer


SeqspecDumper.add_representer(
    Assay,
    represent_tagged("!Assay"),
)
SeqspecDumper.add_representer(
    Read,
    represent_tagged("!Read"),
)
SeqspecDumper.add_representer(
    File,
    represent_tagged("!File"),
)
SeqspecDumper.add_representer(
    Region,
    represent_tagged("!Region"),
)
SeqspecDumper.add_representer(
    Onlist,
    represent_tagged("!Onlist"),
)


def portal_download_filename(
    record: dict,
    successful_download_url: str = "",
) -> str:
    if successful_download_url.startswith(
        ("http://", "https://")
    ):
        filename = Path(
            urlparse(
                successful_download_url
            ).path
        ).name
        if filename:
            return filename

    accession = str(
        record.get("accession") or ""
    )
    file_format = str(
        record.get("file_format") or ""
    ).lower()

    if file_format == "tsv":
        suffix = ".tsv.gz"
    elif file_format:
        suffix = "." + file_format
    else:
        suffix = ".tsv.gz"

    return accession + suffix


def portal_download_url(
    record: dict,
    filename: str,
) -> str:
    portal_id = str(
        record.get("@id") or ""
    )
    if not portal_id:
        raise RuntimeError(
            "Portal record has no @id"
        )

    return (
        "https://api.data.igvf.org"
        + portal_id
        + "@@download/"
        + filename
    )


def make_portal_file_object(
    row: dict,
) -> File:
    accession = str(
        row["accession"]
    )
    filename = (
        accession + ".fastq.gz"
    )
    portal_id = str(
        row["portal_id"]
    )

    if not portal_id:
        raise RuntimeError(
            f"SequenceFile {accession} has no Portal @id"
        )

    return File(
        {
            "file_id": accession,
            "filename": filename,
            "filetype": "fastq",
            "filesize": int(
                row["file_size"]
            ),
            "url": (
                "https://api.data.igvf.org"
                + portal_id
                + "@@download/"
                + filename
            ),
            "urltype": "https",
            "md5": str(
                row["portal_md5sum"]
            ),
        }
    )


def make_local_file_object(
    row: dict,
) -> File:
    accession = str(
        row["accession"]
    )
    local_path = Path(
        row["path"]
    ).resolve()

    if not local_path.is_file():
        raise FileNotFoundError(
            local_path
        )

    expected_md5 = str(
        row["portal_md5sum"]
    )
    if expected_md5:
        observed_md5 = str(
            row.get("local_md5sum", "")
        ).strip()
        if (
            observed_md5
            and observed_md5 != expected_md5
        ):
            raise RuntimeError(
                f"Local/Portal MD5 mismatch for {accession}"
            )

    return File(
        {
            "file_id": accession,
            # Absolute path is deliberate: seqspec's local-resource checker
            # joins relative names to the YAML directory, but preserves an
            # absolute filename.
            "filename": str(local_path),
            "filetype": "fastq",
            "filesize": int(
                local_path.stat().st_size
            ),
            "url": str(local_path),
            "urltype": "local",
            "md5": (
                expected_md5
                or md5sum(local_path)
            ),
        }
    )


def make_segment_region(
    *,
    region_id: str,
    segment: dict,
    parent_id: str,
) -> Region:
    length = (
        int(segment["end"])
        - int(segment["start"])
    )

    return Region(
        {
            "region_id": region_id,
            "region_type": "linker",
            "name": region_id,
            "sequence_type": (
                "fixed"
                if segment["kind"]
                == "fixed"
                else "random"
            ),
            "sequence": str(
                segment["sequence"]
            ),
            "min_len": length,
            "max_len": length,
            "onlist": None,
            "regions": None,
            "parent_id": parent_id,
        }
    )


def build_assay(
    *,
    screen: str,
    file_set: str,
    layout: dict,
    onlist: Onlist,
    sequence_files: list[File],
    read_min_len: int,
    read_max_len: int,
) -> Assay:
    root_region_id = "crispr"

    primer_region = Region(
        {
            "region_id": (
                "crispr_custom_read1_primer"
            ),
            "region_type": (
                "custom_primer"
            ),
            "name": (
                "Custom guide-amplicon "
                "Read 1 primer"
            ),
            "sequence_type": "fixed",
            "sequence": (
                CUSTOM_READ1_PRIMER
            ),
            "min_len": len(
                CUSTOM_READ1_PRIMER
            ),
            "max_len": len(
                CUSTOM_READ1_PRIMER
            ),
            "onlist": None,
            "regions": None,
            "parent_id": root_region_id,
        }
    )

    child_regions: list[Region] = [
        primer_region
    ]
    sequenced_region_sequences: list[
        str
    ] = []

    for index, segment in enumerate(
        layout["prefix_segments"],
        start=1,
    ):
        region = make_segment_region(
            region_id=(
                "crispr_read_prefix_"
                f"{index}"
            ),
            segment=segment,
            parent_id=root_region_id,
        )
        child_regions.append(region)
        sequenced_region_sequences.append(
            str(segment["sequence"])
        )

    guide_length = int(
        layout["guide_length"]
    )

    guide_region = Region(
        {
            "region_id": (
                "crispr_guide_target"
            ),
            # This is allowed by the IGVF-specific schema. The default
            # upstream schema is not the correct validation target here.
            "region_type": (
                "sgrna_target"
            ),
            "name": (
                "CRISPR guide spacer"
            ),
            "sequence_type": "onlist",
            "sequence": (
                "N" * guide_length
            ),
            "min_len": guide_length,
            "max_len": guide_length,
            "onlist": onlist,
            "regions": None,
            "parent_id": root_region_id,
        }
    )
    child_regions.append(guide_region)
    sequenced_region_sequences.append(
        "N" * guide_length
    )

    for index, segment in enumerate(
        layout["suffix_segments"],
        start=1,
    ):
        region = make_segment_region(
            region_id=(
                "crispr_read_suffix_"
                f"{index}"
            ),
            segment=segment,
            parent_id=root_region_id,
        )
        child_regions.append(region)
        sequenced_region_sequences.append(
            str(segment["sequence"])
        )

    sequenced_sequence = "".join(
        sequenced_region_sequences
    )
    library_sequence = (
        CUSTOM_READ1_PRIMER
        + sequenced_sequence
    )

    root_region = Region(
        {
            "region_id": root_region_id,
            "region_type": "crispr",
            "name": (
                "CRISPR guide amplicon"
            ),
            "sequence_type": "joined",
            "sequence": library_sequence,
            "min_len": len(
                library_sequence
            ),
            "max_len": len(
                library_sequence
            ),
            "onlist": None,
            "regions": child_regions,
            "parent_id": None,
        }
    )

    read = Read(
        {
            "read_id": (
                f"{file_set}_read_1"
            ),
            "name": "Read 1",
            "modality": "crispr",
            "primer_id": (
                "crispr_custom_read1_primer"
            ),
            "min_len": int(
                read_min_len
            ),
            "max_len": int(
                read_max_len
            ),
            "strand": "pos",
            "files": sequence_files,
        }
    )

    config = SCREEN_CONFIG[
        screen
    ]

    return Assay(
        {
            "seqspec_version": (
                SEQSPEC_VERSION
            ),
            "assay_id": file_set,
            "name": (
                config["display_name"]
                + " guide amplicon"
            ),
            "doi": "none",
            "date": config[
                "sequence_date"
            ],
            "description": config[
                "description"
            ],
            "modalities": ["crispr"],
            "lib_struct": (
                "https://igvf.github.io/"
                "seqspec/"
            ),
            "library_protocol": (
                LIBRARY_PROTOCOL
            ),
            "library_kit": (
                LIBRARY_KIT
            ),
            "sequence_protocol": (
                SEQUENCE_PROTOCOL
            ),
            "sequence_kit": (
                SEQUENCE_KIT
            ),
            "sequence_spec": [read],
            "library_spec": [
                root_region
            ],
        }
    )


generated_yaml_rows: list[dict] = []
seqspec_groups: dict[
    tuple[str, str],
    dict,
] = {}

for (
    screen,
    file_set,
), group in (
    canonical_sequence_files.groupby(
        ["screen", "file_set"]
    )
):
    layout = group_layouts[
        (screen, file_set)
    ]
    group = group.sort_values(
        ["accession"]
    )

    guide_record = (
        guide_portal_records[screen]
    )
    guide_filename = (
        portal_download_filename(
            guide_record,
            guide_download_urls[
                screen
            ],
        )
    )
    guide_local_path = Path(
        guide_library_inventory.loc[
            guide_library_inventory[
                "screen"
            ].eq(screen),
            "local_path",
        ].iloc[0]
    ).resolve()

    if not guide_local_path.is_file():
        raise FileNotFoundError(
            guide_local_path
        )

    portal_onlist = Onlist(
        {
            "file_id": (
                GUIDE_LIBRARY_ACCESSIONS[
                    screen
                ]
            ),
            "filename": guide_filename,
            "filetype": "tsv",
            "filesize": int(
                guide_record[
                    "file_size"
                ]
            ),
            # Use the exact URL that already succeeded during download.
            "url": guide_download_urls[
                screen
            ],
            "urltype": "https",
            "md5": str(
                guide_record["md5sum"]
            ),
        }
    )

    local_onlist = Onlist(
        {
            "file_id": (
                GUIDE_LIBRARY_ACCESSIONS[
                    screen
                ]
            ),
            "filename": str(
                guide_local_path
            ),
            "filetype": "tsv",
            "filesize": int(
                guide_local_path.stat().st_size
            ),
            "url": str(
                guide_local_path
            ),
            "urltype": "local",
            "md5": (
                str(
                    guide_record.get(
                        "md5sum",
                        "",
                    )
                )
                or md5sum(
                    guide_local_path
                )
            ),
        }
    )

    modal_length = int(
        layout[
            "modal_read_length"
        ]
    )

    min_length_values = pd.to_numeric(
        group[
            "minimum_read_length"
        ].replace("", np.nan),
        errors="coerce",
    ).dropna()
    max_length_values = pd.to_numeric(
        group[
            "maximum_read_length"
        ].replace("", np.nan),
        errors="coerce",
    ).dropna()

    read_min_len = (
        int(
            min_length_values.min()
        )
        if not min_length_values.empty
        else modal_length
    )
    read_max_len = (
        int(
            max_length_values.max()
        )
        if not max_length_values.empty
        else modal_length
    )

    portal_sequence_files = [
        make_portal_file_object(row)
        for row in group.to_dict(
            "records"
        )
    ]
    local_sequence_files = [
        make_local_file_object(row)
        for row in group.to_dict(
            "records"
        )
    ]

    portal_assay = build_assay(
        screen=screen,
        file_set=file_set,
        layout=layout,
        onlist=portal_onlist,
        sequence_files=(
            portal_sequence_files
        ),
        read_min_len=read_min_len,
        read_max_len=read_max_len,
    )
    local_assay = build_assay(
        screen=screen,
        file_set=file_set,
        layout=layout,
        onlist=local_onlist,
        sequence_files=(
            local_sequence_files
        ),
        read_min_len=read_min_len,
        read_max_len=read_max_len,
    )

    stem = (
        f"{screen}_{file_set}_"
        "guide_amplicon_seqspec"
    )
    portal_draft_path = (
        YAML_DIR
        / f"{stem}.draft.yaml"
    )
    portal_final_path = (
        YAML_DIR
        / f"{stem}.yaml"
    )
    local_draft_path = (
        YAML_DIR
        / f"{stem}.local_validation.draft.yaml"
    )
    local_final_path = (
        YAML_DIR
        / f"{stem}.local_validation.yaml"
    )

    portal_draft_path.write_text(
        yaml.dump(
            portal_assay,
            Dumper=SeqspecDumper,
            sort_keys=False,
            allow_unicode=True,
            default_flow_style=False,
            width=120,
        ),
        encoding="utf-8",
    )
    local_draft_path.write_text(
        yaml.dump(
            local_assay,
            Dumper=SeqspecDumper,
            sort_keys=False,
            allow_unicode=True,
            default_flow_style=False,
            width=120,
        ),
        encoding="utf-8",
    )

    document_alias = (
        "jesse-engreitz:"
        "document_WTC11-EC_"
        f"{screen}_{file_set}_"
        "guide_amplicon_seqspec_v1"
    )

    portal_resource_urls = [
        str(
            portal_onlist["url"]
        )
    ] + [
        str(
            file_object["url"]
        )
        for file_object in (
            portal_sequence_files
        )
    ]
    portal_resource_filenames = [
        str(
            portal_onlist["filename"]
        )
    ] + [
        str(
            file_object["filename"]
        )
        for file_object in (
            portal_sequence_files
        )
    ]

    seqspec_groups[
        (screen, file_set)
    ] = {
        "screen": screen,
        "file_set": file_set,
        "draft_yaml": (
            portal_draft_path
        ),
        "final_yaml": (
            portal_final_path
        ),
        "local_validation_draft_yaml": (
            local_draft_path
        ),
        "local_validation_final_yaml": (
            local_final_path
        ),
        "document_alias": (
            document_alias
        ),
        "sequence_file_accessions": (
            group[
                "accession"
            ].tolist()
        ),
        "layout": layout,
        "portal_resource_urls": (
            portal_resource_urls
        ),
        "portal_resource_filenames": (
            portal_resource_filenames
        ),
    }

    generated_yaml_rows.append(
        {
            "screen": screen,
            "file_set": file_set,
            "sequence_files": len(
                group
            ),
            "document_alias": (
                document_alias
            ),
            "draft_yaml": str(
                portal_draft_path
            ),
            "final_yaml": str(
                portal_final_path
            ),
            "local_validation_draft_yaml": str(
                local_draft_path
            ),
            "local_validation_final_yaml": str(
                local_final_path
            ),
            "guide_library_accession": (
                GUIDE_LIBRARY_ACCESSIONS[
                    screen
                ]
            ),
            "guide_length": int(
                layout["guide_length"]
            ),
            "guide_offset": int(
                layout["offset"]
            ),
            "modal_read_length": (
                modal_length
            ),
            "match_fraction": float(
                layout[
                    "match_fraction"
                ]
            ),
            "inferred_read_sequence": (
                "".join(
                    str(segment["sequence"])
                    for segment in (
                        layout[
                            "prefix_segments"
                        ]
                    )
                )
                + (
                    "N"
                    * int(
                        layout[
                            "guide_length"
                        ]
                    )
                )
                + "".join(
                    str(segment["sequence"])
                    for segment in (
                        layout[
                            "suffix_segments"
                        ]
                    )
                )
            ),
        }
    )

generated_yaml_inventory = (
    pd.DataFrame(
        generated_yaml_rows
    )
)
GENERATED_YAML_INVENTORY_PATH = (
    WORK_DIR
    / "generated_seqspec_inventory.tsv"
)
write_tsv(
    generated_yaml_inventory,
    GENERATED_YAML_INVENTORY_PATH,
)

print(
    generated_yaml_inventory.to_string(
        index=False
    )
)
print(
    "\nSaved:",
    GENERATED_YAML_INVENTORY_PATH,
)

## 10. Install or select a compatible seqspec executable, then validate

Seqspec 0.4.0 requires Python 3.12 or newer. On Sherlock, also pin NumPy
2.2.6 so pip uses a glibc-2.17-compatible wheel instead of trying to compile
the newer NumPy source distribution.

```bash
ENV=/oak/stanford/groups/engreitz/Users/tri/envs/seqspec-0.4
CONSTRAINTS="$ENV/seqspec-sherlock-constraints.txt"

mamba create -p "$ENV" python=3.12 pip -y

printf 'numpy==2.2.6\n' > "$CONSTRAINTS"

"$ENV/bin/python" -m pip install --upgrade pip

"$ENV/bin/python" -m pip install       --no-cache-dir       --only-binary=:all:       --constraint "$CONSTRAINTS"       "seqspec==0.4.0"
```

The environment already created by the failed attempt can be repaired; it
does not need to be deleted. The next bootstrap cell performs the same
constrained installation after `INSTALL_SEQSPEC_0_4_NOW = True`.

The CLI help may display only `SKIP`; that is expected. Capability is tested
by direct command execution and Python-level filter semantics, not by
searching the help text for the two mode names.


### 10A. Confirm the isolated Matplotlib environment

This read-only check shows the Jupyter backend inherited by the current
kernel and the backend that will be supplied to external seqspec
commands. The external value must be `Agg`.

In [ ]:
current_backend = os.environ.get(
    "MPLBACKEND",
    "<not set>",
)
external_environment = (
    seqspec_subprocess_env()
)

backend_diagnostic = pd.DataFrame(
    [
        {
            "context": "current Jupyter kernel",
            "MPLBACKEND": current_backend,
            "MPLCONFIGDIR": os.environ.get(
                "MPLCONFIGDIR",
                "<not set>",
            ),
        },
        {
            "context": "external seqspec subprocess",
            "MPLBACKEND": external_environment[
                "MPLBACKEND"
            ],
            "MPLCONFIGDIR": external_environment[
                "MPLCONFIGDIR"
            ],
        },
    ]
)

BACKEND_DIAGNOSTIC_PATH = (
    WORK_DIR
    / "seqspec_matplotlib_environment.tsv"
)
write_tsv(
    backend_diagnostic,
    BACKEND_DIAGNOSTIC_PATH,
)

print(
    backend_diagnostic.to_string(
        index=False
    )
)
print(
    "\nSaved:",
    BACKEND_DIAGNOSTIC_PATH,
)

if (
    external_environment[
        "MPLBACKEND"
    ].lower()
    != "agg"
):
    raise RuntimeError(
        "External seqspec backend is not Agg"
    )

### 10B. Optional bootstrap of seqspec 0.4.0

Set `INSTALL_SEQSPEC_0_4_NOW = True` in Section 1 and run this cell once.
The existing seqspec 0.3.1 prefix is left unchanged.

In [ ]:
def find_mamba_executable() -> Path:
    candidates: list[Path] = []

    if MAMBA_EXECUTABLE_OVERRIDE:
        candidates.append(
            Path(
                MAMBA_EXECUTABLE_OVERRIDE
            )
        )

    environment_mamba = os.environ.get(
        "MAMBA_EXE",
        "",
    )
    if environment_mamba:
        candidates.append(
            Path(environment_mamba)
        )

    for name in (
        "mamba",
        "micromamba",
    ):
        resolved = shutil.which(name)
        if resolved:
            candidates.append(
                Path(resolved)
            )

    home = Path.home()
    candidates.extend(
        [
            home / "mambaforge/bin/mamba",
            home / "miniforge3/bin/mamba",
            home / "miniconda3/bin/mamba",
            home / "micromamba/bin/micromamba",
        ]
    )

    seen: set[str] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        "No mamba or micromamba executable was found. "
        "Set MAMBA_EXECUTABLE_OVERRIDE in Section 1."
    )


def run_install_command(
    command: list[str],
    label: str,
) -> subprocess.CompletedProcess:
    print("\n" + label)
    print(
        "Command:",
        " ".join(command),
    )

    completed = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=seqspec_subprocess_env(),
        check=False,
    )
    print(completed.stdout)

    log_path = (
        LOG_DIR
        / (
            "seqspec_environment_"
            + safe_stem(label)
            + ".log"
        )
    )
    log_path.write_text(
        completed.stdout,
        encoding="utf-8",
    )

    if completed.returncode != 0:
        raise RuntimeError(
            f"{label} failed; review {log_path}"
        )

    return completed


if not INSTALL_SEQSPEC_0_4_NOW:
    print(
        "Installation is disabled. Set "
        "INSTALL_SEQSPEC_0_4_NOW=True in Section 1 and rerun.\n\n"
        "Equivalent repair commands:\n\n"
        f"ENV={PREFERRED_SEQSPEC_ENV}\n"
        'CONSTRAINTS="$ENV/seqspec-sherlock-constraints.txt"\n'
        f"printf 'numpy=={SEQSPEC_NUMPY_PIN}\\n' > "
        '"$CONSTRAINTS"\n'
        '"$ENV/bin/python" -m pip install --upgrade pip\n'
        '"$ENV/bin/python" -m pip install '
        "--no-cache-dir --only-binary=:all: "
        '--constraint "$CONSTRAINTS" '
        '"seqspec==0.4.0"'
    )
else:
    mamba_executable = (
        find_mamba_executable()
    )
    environment_python = (
        PREFERRED_SEQSPEC_ENV
        / "bin/python"
    )
    environment_seqspec = (
        PREFERRED_SEQSPEC_ENV
        / "bin/seqspec"
    )

    if not environment_python.is_file():
        run_install_command(
            [
                str(mamba_executable),
                "create",
                "-p",
                str(
                    PREFERRED_SEQSPEC_ENV
                ),
                "python=3.12",
                "pip",
                "-y",
            ],
            "create_seqspec_0_4_environment",
        )
    else:
        version_check = subprocess.run(
            [
                str(environment_python),
                "-c",
                (
                    "import sys; "
                    "print('.'.join(map(str, sys.version_info[:3]))); "
                    "raise SystemExit("
                    "0 if sys.version_info >= (3, 12) else 1)"
                ),
            ],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=False,
        )
        print(
            "Existing dedicated environment Python:",
            version_check.stdout.strip(),
        )
        if version_check.returncode != 0:
            raise RuntimeError(
                f"{PREFERRED_SEQSPEC_ENV} exists but uses Python "
                "older than 3.12. Move that incomplete prefix aside "
                "and rerun; it is not deleted automatically."
            )

    constraints_path = (
        PREFERRED_SEQSPEC_ENV
        / "seqspec-sherlock-constraints.txt"
    )
    constraints_path.write_text(
        f"numpy=={SEQSPEC_NUMPY_PIN}\n",
        encoding="utf-8",
    )
    print(
        "\nConstraint file:",
        constraints_path,
    )
    print(
        constraints_path.read_text(
            encoding="utf-8"
        )
    )

    # Keep pip current enough to recognize manylinux2014 wheels.
    run_install_command(
        [
            str(environment_python),
            "-m",
            "pip",
            "install",
            "--upgrade",
            "pip",
        ],
        "upgrade_pip",
    )

    # Wheel-only resolution is deliberate: if any dependency lacks a
    # Sherlock-compatible wheel, stop clearly instead of compiling on the
    # login node.
    run_install_command(
        [
            str(environment_python),
            "-m",
            "pip",
            "install",
            "--upgrade",
            "--no-cache-dir",
            "--only-binary=:all:",
            "--constraint",
            str(constraints_path),
            "seqspec==0.4.0",
        ],
        "install_seqspec_0_4_0_with_sherlock_constraints",
    )

    run_install_command(
        [
            str(environment_python),
            "-m",
            "pip",
            "check",
        ],
        "pip_check_seqspec_environment",
    )

    if not environment_seqspec.is_file():
        raise FileNotFoundError(
            environment_seqspec
        )


        help_result = run_install_command(
            [
                str(environment_seqspec),
                "check",
                "--help",
            ],
            "verify_seqspec_check_help",
        )

        if "--skip" not in help_result.stdout:
            raise RuntimeError(
                "Installed seqspec does not expose the --skip option"
            )

        # Argparse uses the explicit metavar SKIP, so mode names are not
        # necessarily printed in the help output. Probe both values directly.
        mode_probe_results = {}
        for skip_mode in (
            "igvf",
            "igvf_onlist_skip",
        ):
            mode_result = run_install_command(
                [
                    str(environment_seqspec),
                    "check",
                    "--skip",
                    skip_mode,
                    "--help",
                ],
                (
                    "verify_seqspec_skip_mode_"
                    + skip_mode
                ),
            )
            mode_probe_results[
                skip_mode
            ] = mode_result.returncode == 0

        package_probe_code = r"""
import json
from pathlib import Path

import numpy
import seqspec
from seqspec.seqspec_check import filter_errors

package_path = Path(seqspec.__file__).resolve()
schema_path = package_path.parent / "schema" / "seqspec.schema.json"
schema_text = schema_path.read_text(encoding="utf-8")

getter = getattr(seqspec, "get_version", None)
version = (
    getter()
    if callable(getter)
    else getattr(seqspec, "__version__", "")
)

synthetic = [
    {
        "severity": "error",
        "error_type": "check_schema",
        "error_message": "synthetic schema error",
        "error_object": "'lib_struct'",
    },
    {
        "severity": "error",
        "error_type": "check_onlist_files_exist",
        "error_message": "synthetic onlist error",
        "error_object": "onlist",
    },
    {
        "severity": "error",
        "error_type": "check_unique_modalities",
        "error_message": "synthetic structural error",
        "error_object": "modalities",
    },
]

igvf_result = filter_errors(
    list(synthetic),
    "igvf",
)
igvf_onlist_skip_result = filter_errors(
    list(synthetic),
    "igvf_onlist_skip",
)

igvf_types = [
    item["error_type"]
    for item in igvf_result
]
igvf_onlist_skip_types = [
    item["error_type"]
    for item in igvf_onlist_skip_result
]

supports_igvf_filter = (
    "check_schema" not in igvf_types
    and "check_onlist_files_exist" in igvf_types
    and "check_unique_modalities" in igvf_types
)

supports_igvf_onlist_skip_filter = (
    "check_schema" not in igvf_onlist_skip_types
    and "check_onlist_files_exist"
    not in igvf_onlist_skip_types
    and "check_unique_modalities"
    in igvf_onlist_skip_types
)

print(
    json.dumps(
        {
            "seqspec_version": str(version),
            "numpy_version": str(numpy.__version__),
            "package": str(package_path),
            "schema": str(schema_path),
            "has_sgrna_target": (
                '"sgrna_target"' in schema_text
            ),
            "supports_igvf_filter": (
                supports_igvf_filter
            ),
            "supports_igvf_onlist_skip_filter": (
                supports_igvf_onlist_skip_filter
            ),
            "igvf_result_types": igvf_types,
            "igvf_onlist_skip_result_types": (
                igvf_onlist_skip_types
            ),
        }
    )
)
"""

        package_probe = run_install_command(
            [
                str(environment_python),
                "-c",
                package_probe_code,
            ],
            "verify_seqspec_numpy_schema_and_filters",
        )

        try:
            package_info = json.loads(
                package_probe.stdout.strip().splitlines()[-1]
            )
        except Exception as error:
            raise RuntimeError(
                "Could not parse the seqspec package capability probe: "
                f"{type(error).__name__}: {error}\n"
                f"Output:\n{package_probe.stdout}"
            ) from error

        if (
            str(
                package_info.get(
                    "numpy_version",
                    "",
                )
            )
            != SEQSPEC_NUMPY_PIN
        ):
            raise RuntimeError(
                "The seqspec environment does not use the required "
                f"NumPy pin {SEQSPEC_NUMPY_PIN}: {package_info}"
            )

        if not bool(
            package_info.get(
                "has_sgrna_target",
                False,
            )
        ):
            raise RuntimeError(
                "The installed seqspec schema lacks sgrna_target"
            )

        if not bool(
            package_info.get(
                "supports_igvf_filter",
                False,
            )
        ):
            raise RuntimeError(
                "The installed seqspec package does not implement the "
                "expected igvf filter behavior"
            )

        if not bool(
            package_info.get(
                "supports_igvf_onlist_skip_filter",
                False,
            )
        ):
            raise RuntimeError(
                "The installed seqspec package does not implement the "
                "expected igvf_onlist_skip filter behavior"
            )
    print(
        "\nDedicated seqspec environment is ready:",
        environment_seqspec,
    )
    print(
        "Rerun the next validation cell. "
        "No kernel restart is required."
    )

In [ ]:
def python_for_seqspec_executable(
    executable: Path,
) -> Path | None:
    adjacent = executable.parent / "python"
    if adjacent.is_file():
        return adjacent

    try:
        first_line = executable.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()[0]
    except Exception:
        first_line = ""

    if first_line.startswith("#!"):
        shebang = Path(
            first_line[2:].strip()
        )
        if shebang.is_file():
            return shebang

    return None


def probe_seqspec_candidate(
        executable: Path,
    ) -> dict:
        result = {
            "executable": str(executable),
            "exists": executable.is_file(),
            "check_help_returncode": "",
            "supports_skip": False,
            "igvf_cli_probe_returncode": "",
            "igvf_onlist_skip_cli_probe_returncode": "",
            "supports_igvf": False,
            "supports_igvf_onlist_skip": False,
            "supports_auth_profile": False,
            "python_executable": "",
            "package_version": "",
            "package_path": "",
            "schema_path": "",
            "schema_has_sgrna_target": False,
            "accepted": False,
            "reason": "",
            "help_preview": "",
            "package_probe_output": "",
        }

        if not executable.is_file():
            result["reason"] = (
                "executable does not exist"
            )
            return result

        help_completed = subprocess.run(
            [
                str(executable),
                "check",
                "--help",
            ],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=seqspec_subprocess_env(),
            check=False,
        )
        help_text = help_completed.stdout

        result["check_help_returncode"] = (
            help_completed.returncode
        )
        result["supports_skip"] = (
            "--skip" in help_text
            or re.search(
                r"(^|\s)-s([,\s]|$)",
                help_text,
            )
            is not None
        )
        result["supports_auth_profile"] = (
            "--auth-profile" in help_text
        )
        result["help_preview"] = " | ".join(
            help_text.splitlines()[:20]
        )

        # Do not infer accepted values from help text: argparse's explicit
        # metavar may hide the choices. Probe each value directly.
        igvf_cli_probe = subprocess.run(
            [
                str(executable),
                "check",
                "--skip",
                "igvf",
                "--help",
            ],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=seqspec_subprocess_env(),
            check=False,
        )
        igvf_onlist_cli_probe = subprocess.run(
            [
                str(executable),
                "check",
                "--skip",
                "igvf_onlist_skip",
                "--help",
            ],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=seqspec_subprocess_env(),
            check=False,
        )

        result[
            "igvf_cli_probe_returncode"
        ] = igvf_cli_probe.returncode
        result[
            "igvf_onlist_skip_cli_probe_returncode"
        ] = igvf_onlist_cli_probe.returncode

        python_executable = (
            python_for_seqspec_executable(
                executable
            )
        )

        filter_capabilities = {
            "supports_igvf_filter": False,
            "supports_igvf_onlist_skip_filter": False,
        }

        if python_executable is not None:
            result["python_executable"] = str(
                python_executable
            )

            package_probe_code = r"""
import json
from pathlib import Path

import seqspec
from seqspec.seqspec_check import filter_errors

package_path = Path(seqspec.__file__).resolve()
schema_path = package_path.parent / "schema" / "seqspec.schema.json"

getter = getattr(seqspec, "get_version", None)
version = ""
if callable(getter):
    try:
        version = str(getter())
    except Exception:
        version = ""
if not version:
    version = str(
        getattr(seqspec, "__version__", "")
    )

schema_text = (
    schema_path.read_text(encoding="utf-8")
    if schema_path.is_file()
    else ""
)

synthetic = [
    {
        "severity": "error",
        "error_type": "check_schema",
        "error_message": "synthetic schema error",
        "error_object": "'lib_struct'",
    },
    {
        "severity": "error",
        "error_type": "check_onlist_files_exist",
        "error_message": "synthetic onlist error",
        "error_object": "onlist",
    },
    {
        "severity": "error",
        "error_type": "check_unique_modalities",
        "error_message": "synthetic structural error",
        "error_object": "modalities",
    },
]

igvf_types = [
    item["error_type"]
    for item in filter_errors(
        list(synthetic),
        "igvf",
    )
]
igvf_onlist_skip_types = [
    item["error_type"]
    for item in filter_errors(
        list(synthetic),
        "igvf_onlist_skip",
    )
]

supports_igvf_filter = (
    "check_schema" not in igvf_types
    and "check_onlist_files_exist" in igvf_types
    and "check_unique_modalities" in igvf_types
)
supports_igvf_onlist_skip_filter = (
    "check_schema" not in igvf_onlist_skip_types
    and "check_onlist_files_exist"
    not in igvf_onlist_skip_types
    and "check_unique_modalities"
    in igvf_onlist_skip_types
)

print(
    json.dumps(
        {
            "version": version,
            "package_path": str(package_path),
            "schema_path": str(schema_path),
            "schema_exists": schema_path.is_file(),
            "schema_has_sgrna_target": (
                '"sgrna_target"' in schema_text
            ),
            "supports_igvf_filter": (
                supports_igvf_filter
            ),
            "supports_igvf_onlist_skip_filter": (
                supports_igvf_onlist_skip_filter
            ),
            "igvf_result_types": igvf_types,
            "igvf_onlist_skip_result_types": (
                igvf_onlist_skip_types
            ),
        }
    )
)
"""

            package_completed = subprocess.run(
                [
                    str(python_executable),
                    "-c",
                    package_probe_code,
                ],
                text=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                env=seqspec_subprocess_env(),
                check=False,
            )
            result["package_probe_output"] = (
                package_completed.stdout.strip()
            )

            if package_completed.returncode == 0:
                try:
                    package_info = json.loads(
                        package_completed.stdout.strip().splitlines()[-1]
                    )
                except Exception:
                    package_info = {}

                result["package_version"] = str(
                    package_info.get(
                        "version",
                        "",
                    )
                )
                result["package_path"] = str(
                    package_info.get(
                        "package_path",
                        "",
                    )
                )
                result["schema_path"] = str(
                    package_info.get(
                        "schema_path",
                        "",
                    )
                )
                result[
                    "schema_has_sgrna_target"
                ] = bool(
                    package_info.get(
                        "schema_has_sgrna_target",
                        False,
                    )
                )
                filter_capabilities[
                    "supports_igvf_filter"
                ] = bool(
                    package_info.get(
                        "supports_igvf_filter",
                        False,
                    )
                )
                filter_capabilities[
                    "supports_igvf_onlist_skip_filter"
                ] = bool(
                    package_info.get(
                        "supports_igvf_onlist_skip_filter",
                        False,
                    )
                )

        result["supports_igvf"] = (
            igvf_cli_probe.returncode == 0
            and filter_capabilities[
                "supports_igvf_filter"
            ]
        )
        result[
            "supports_igvf_onlist_skip"
        ] = (
            igvf_onlist_cli_probe.returncode == 0
            and filter_capabilities[
                "supports_igvf_onlist_skip_filter"
            ]
        )

        accepted = all(
            [
                help_completed.returncode == 0,
                result["supports_skip"],
                result["supports_igvf"],
                result[
                    "supports_igvf_onlist_skip"
                ],
                result[
                    "schema_has_sgrna_target"
                ],
            ]
        )
        result["accepted"] = accepted

        if accepted:
            result["reason"] = "compatible"
        else:
            missing = []
            if help_completed.returncode != 0:
                missing.append(
                    "check --help failed"
                )
            if not result["supports_skip"]:
                missing.append(
                    "--skip is unavailable"
                )
            if (
                igvf_cli_probe.returncode
                != 0
            ):
                missing.append(
                    "CLI rejected --skip igvf"
                )
            elif not filter_capabilities[
                "supports_igvf_filter"
            ]:
                missing.append(
                    "igvf filter semantics are unavailable"
                )
            if (
                igvf_onlist_cli_probe.returncode
                != 0
            ):
                missing.append(
                    "CLI rejected --skip igvf_onlist_skip"
                )
            elif not filter_capabilities[
                "supports_igvf_onlist_skip_filter"
            ]:
                missing.append(
                    "igvf_onlist_skip filter semantics are unavailable"
                )
            if not result[
                "schema_has_sgrna_target"
            ]:
                missing.append(
                    "schema lacks sgrna_target"
                )
            result["reason"] = "; ".join(
                missing
            )

        return result

def find_seqspec_executable() -> tuple[
    Path,
    dict,
]:
    candidates: list[Path] = []

    if SEQSPEC_EXECUTABLE_OVERRIDE:
        candidates.append(
            Path(
                SEQSPEC_EXECUTABLE_OVERRIDE
            )
        )

    candidates.extend(
        [
            PREFERRED_SEQSPEC_ENV
            / "bin/seqspec",
            LEGACY_SEQSPEC_ENV
            / "bin/seqspec",
        ]
    )

    path_executable = shutil.which(
        "seqspec"
    )
    if path_executable:
        candidates.append(
            Path(path_executable)
        )

    unique_candidates: list[Path] = []
    seen: set[str] = set()

    for candidate in candidates:
        candidate = candidate.expanduser()
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        unique_candidates.append(
            candidate
        )

    audit_rows = [
        probe_seqspec_candidate(
            candidate
        )
        for candidate in unique_candidates
    ]
    audit = pd.DataFrame(
        audit_rows
    )

    audit_path = (
        WORK_DIR
        / "seqspec_executable_capability_audit.tsv"
    )
    write_tsv(
        audit,
        audit_path,
    )

    print(
        audit[
            [
                "executable",
                "exists",
                "check_help_returncode",
                "package_version",
                "supports_skip",
                "igvf_cli_probe_returncode",
                "igvf_onlist_skip_cli_probe_returncode",
                "supports_igvf",
                "supports_igvf_onlist_skip",
                "schema_has_sgrna_target",
                "accepted",
                "reason",
            ]
        ].to_string(index=False)
    )
    print(
        "\nSaved:",
        audit_path,
    )

    accepted = audit.loc[
        audit["accepted"].astype(bool)
    ]

    if accepted.empty:
        raise RuntimeError(
            "No IGVF-capable seqspec executable was found. "
            "Create /oak/stanford/groups/engreitz/Users/tri/"
            "envs/seqspec-0.4 with seqspec==0.4.0, then rerun. "
            f"Review {audit_path}"
        )

    selected = accepted.iloc[0].to_dict()
    return (
        Path(
            selected["executable"]
        ),
        selected,
    )

def run_logged_command(
    command: list[str],
    log_path: Path,
) -> subprocess.CompletedProcess:
    completed = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=seqspec_subprocess_env(),
        check=False,
    )
    log_path.write_text(
        completed.stdout,
        encoding="utf-8",
    )
    return completed


def seqspec_error_lines(
    output: str,
) -> list[str]:
    return [
        line.strip()
        for line in output.splitlines()
        if re.search(
            r"\[error(?:\s+\d+)?\]",
            line,
            flags=re.IGNORECASE,
        )
    ]


def seqspec_warning_lines(
    output: str,
) -> list[str]:
    return [
        line.strip()
        for line in output.splitlines()
        if re.search(
            r"\[warning(?:\s+\d+)?\]",
            line,
            flags=re.IGNORECASE,
        )
    ]


def assert_seqspec_check_invoked(
    completed: subprocess.CompletedProcess,
    log_path: Path,
    *,
    label: str,
) -> None:
    output = completed.stdout

    invocation_failure = (
        completed.returncode == 2
        or re.search(
            (
                r"(^|\n)usage:"
                r"|unrecognized arguments"
                r"|invalid choice"
                r"|the following arguments are required"
            ),
            output,
            flags=re.IGNORECASE,
        )
        is not None
    )

    if invocation_failure:
        preview = " | ".join(
            output.splitlines()[:30]
        )
        raise RuntimeError(
            f"{label} did not run because the selected seqspec "
            f"CLI rejected the command line. Executable: "
            f"{SEQSPEC_EXECUTABLE}. Output: {preview}. "
            f"Full log: {log_path}. Executable audit: "
            f"{SEQSPEC_EXECUTABLE_AUDIT_PATH}"
        )


def authenticated_range_probe(
    url: str,
    *,
    timeout: int = 90,
) -> dict:
    try:
        response = requests.get(
            url,
            auth=portal_auth(),
            headers={
                "Range": "bytes=0-0",
                "Accept": "*/*",
            },
            stream=True,
            allow_redirects=True,
            timeout=timeout,
        )
        status_code = int(
            response.status_code
        )
        final_url = str(
            response.url
        )
        content_range = str(
            response.headers.get(
                "Content-Range",
                "",
            )
        )
        ok = status_code in {
            200,
            206,
        }
        response.close()
        return {
            "url": url,
            "final_url": final_url,
            "status_code": status_code,
            "content_range": (
                content_range
            ),
            "success": ok,
            "error": "",
        }
    except Exception as error:
        return {
            "url": url,
            "final_url": "",
            "status_code": "",
            "content_range": "",
            "success": False,
            "error": (
                f"{type(error).__name__}: "
                f"{error}"
            ),
        }


def representative_resource_urls(
    urls: list[str],
) -> list[str]:
    if not urls:
        return []

    # First URL is the guide onlist. Probe it plus up to three read files.
    selected_indices = {0}
    read_count = len(urls) - 1

    if read_count > 0:
        selected_indices.add(1)
        selected_indices.add(
            1 + read_count // 2
        )
        selected_indices.add(
            len(urls) - 1
        )

    return [
        urls[index]
        for index in sorted(
            selected_indices
        )
        if 0 <= index < len(urls)
    ]


(
    SEQSPEC_EXECUTABLE,
    SELECTED_SEQSPEC_INFO,
) = find_seqspec_executable()

SEQSPEC_EXECUTABLE_AUDIT_PATH = (
    WORK_DIR
    / "seqspec_executable_capability_audit.tsv"
)

print(
    "\nSelected seqspec executable:",
    SEQSPEC_EXECUTABLE,
)
print(
    "Selected seqspec package version:",
    SELECTED_SEQSPEC_INFO.get(
        "package_version",
        "",
    ),
)
print(
    "Selected seqspec package path:",
    SELECTED_SEQSPEC_INFO.get(
        "package_path",
        "",
    ),
)

validation_rows: list[dict] = []
remote_probe_rows: list[dict] = []

for group in seqspec_groups.values():
    portal_draft_path = Path(
        group["draft_yaml"]
    )
    portal_final_path = Path(
        group["final_yaml"]
    )
    local_draft_path = Path(
        group[
            "local_validation_draft_yaml"
        ]
    )
    local_final_path = Path(
        group[
            "local_validation_final_yaml"
        ]
    )

    # Format both versions from the same structure.
    portal_format_log = (
        LOG_DIR
        / (
            portal_final_path.stem
            + "_format.log"
        )
    )
    portal_format_result = (
        run_logged_command(
            [
                str(
                    SEQSPEC_EXECUTABLE
                ),
                "format",
                "-o",
                str(
                    portal_final_path
                ),
                str(
                    portal_draft_path
                ),
            ],
            portal_format_log,
        )
    )

    local_format_log = (
        LOG_DIR
        / (
            local_final_path.stem
            + "_format.log"
        )
    )
    local_format_result = (
        run_logged_command(
            [
                str(
                    SEQSPEC_EXECUTABLE
                ),
                "format",
                "-o",
                str(
                    local_final_path
                ),
                str(
                    local_draft_path
                ),
            ],
            local_format_log,
        )
    )

    if (
        portal_format_result.returncode
        != 0
        or not portal_final_path.is_file()
    ):
        raise RuntimeError(
            "Portal-form seqspec format failed; "
            f"review {portal_format_log}"
        )

    if (
        local_format_result.returncode
        != 0
        or not local_final_path.is_file()
    ):
        raise RuntimeError(
            "Local-validation seqspec format failed; "
            f"review {local_format_log}"
        )

    # Full IGVF schema and local resource validation.
    local_check_log = (
        LOG_DIR
        / (
            local_final_path.stem
            + "_check_igvf_local.log"
        )
    )
    local_check_result = (
        run_logged_command(
            [
                str(
                    SEQSPEC_EXECUTABLE
                ),
                "check",
                "--skip",
                "igvf",
                str(
                    local_final_path
                ),
            ],
            local_check_log,
        )
    )
    assert_seqspec_check_invoked(
        local_check_result,
        local_check_log,
        label="Local IGVF seqspec check",
    )
    local_errors = seqspec_error_lines(
        local_check_result.stdout
    )
    local_warnings = seqspec_warning_lines(
        local_check_result.stdout
    )

    # Final Portal-form validation. The IGVF-specific schema permits
    # sgrna_target; onlist reachability is handled by the successful download
    # and the local mirror.
    portal_check_log = (
        LOG_DIR
        / (
            portal_final_path.stem
            + "_check_igvf_portal.log"
        )
    )
    portal_check_result = (
        run_logged_command(
            [
                str(
                    SEQSPEC_EXECUTABLE
                ),
                "check",
                "--skip",
                "igvf_onlist_skip",
                str(
                    portal_final_path
                ),
            ],
            portal_check_log,
        )
    )
    assert_seqspec_check_invoked(
        portal_check_result,
        portal_check_log,
        label="Portal-form IGVF seqspec check",
    )
    portal_errors = seqspec_error_lines(
        portal_check_result.stdout
    )
    portal_warnings = seqspec_warning_lines(
        portal_check_result.stdout
    )

    expected_filenames = set(
        map(
            str,
            group[
                "portal_resource_filenames"
            ],
        )
    )

    expected_remote_errors: list[str] = []
    unexpected_portal_errors: list[str] = []

    for error_line in portal_errors:
        match = re.search(
            r"\[error\s+\d+\]\s+"
            r"(.+?)\s+does not exist$",
            error_line,
            flags=re.IGNORECASE,
        )
        if (
            match
            and match.group(1)
            in expected_filenames
        ):
            expected_remote_errors.append(
                error_line
            )
        else:
            unexpected_portal_errors.append(
                error_line
            )

    probe_rows_for_group: list[dict] = []
    for url in representative_resource_urls(
        list(
            group[
                "portal_resource_urls"
            ]
        )
    ):
        probe = authenticated_range_probe(
            url
        )
        probe.update(
            {
                "screen": group[
                    "screen"
                ],
                "file_set": group[
                    "file_set"
                ],
            }
        )
        probe_rows_for_group.append(
            probe
        )
        remote_probe_rows.append(
            probe
        )

    remote_probe_passed = bool(
        probe_rows_for_group
    ) and all(
        bool(row["success"])
        for row in probe_rows_for_group
    )

    local_passed = (
        local_check_result.returncode == 0
        and not local_errors
    )
    portal_structure_passed = (
        portal_check_result.returncode == 0
        and not unexpected_portal_errors
    )

    passed = (
        local_passed
        and portal_structure_passed
        and remote_probe_passed
    )

    validation_rows.append(
        {
            "screen": group["screen"],
            "file_set": (
                group["file_set"]
            ),
            "selected_seqspec_executable": str(
                SEQSPEC_EXECUTABLE
            ),
            "selected_seqspec_version": str(
                SELECTED_SEQSPEC_INFO.get(
                    "package_version",
                    "",
                )
            ),
            "selected_seqspec_package_path": str(
                SELECTED_SEQSPEC_INFO.get(
                    "package_path",
                    "",
                )
            ),
            "final_yaml": str(
                portal_final_path
            ),
            "local_validation_yaml": str(
                local_final_path
            ),
            "size_bytes": (
                portal_final_path.stat().st_size
            ),
            "md5sum": md5sum(
                portal_final_path
            ),
            "sha256": sha256sum(
                portal_final_path
            ),
            "portal_format_log": str(
                portal_format_log
            ),
            "local_format_log": str(
                local_format_log
            ),
            "local_check_log": str(
                local_check_log
            ),
            "portal_check_log": str(
                portal_check_log
            ),
            "local_check_returncode": (
                local_check_result.returncode
            ),
            "portal_check_returncode": (
                portal_check_result.returncode
            ),
            "local_error_lines": " | ".join(
                local_errors
            ),
            "local_warning_lines": " | ".join(
                local_warnings
            ),
            "portal_expected_remote_errors": (
                " | ".join(
                    expected_remote_errors
                )
            ),
            "portal_unexpected_errors": (
                " | ".join(
                    unexpected_portal_errors
                )
            ),
            "portal_warning_lines": (
                " | ".join(
                    portal_warnings
                )
            ),
            "authenticated_probe_count": len(
                probe_rows_for_group
            ),
            "authenticated_probe_passed": (
                remote_probe_passed
            ),
            "validation_basis": (
                "capability-audited seqspec CLI + "
                "IGVF local mirror + Portal-form "
                "IGVF schema + authenticated ranged "
                "GET probes"
            ),
            "validation": (
                "PASS"
                if passed
                else "FAIL"
            ),
        }
    )

seqspec_validation = pd.DataFrame(
    validation_rows
)
SEQSPEC_VALIDATION_PATH = (
    WORK_DIR
    / "seqspec_validation.tsv"
)
write_tsv(
    seqspec_validation,
    SEQSPEC_VALIDATION_PATH,
)

remote_probe_results = pd.DataFrame(
    remote_probe_rows
)
REMOTE_PROBE_RESULTS_PATH = (
    WORK_DIR
    / "seqspec_portal_resource_probe.tsv"
)
write_tsv(
    remote_probe_results,
    REMOTE_PROBE_RESULTS_PATH,
)

display_columns = [
    "screen",
    "file_set",
    "selected_seqspec_version",
    "validation",
    "local_check_returncode",
    "portal_check_returncode",
    "authenticated_probe_count",
    "authenticated_probe_passed",
    "portal_unexpected_errors",
]

print(
    seqspec_validation[
        display_columns
    ].to_string(index=False)
)
print(
    "\nPortal resource probes:"
)
print(
    remote_probe_results.to_string(
        index=False
    )
)
print(
    "\nSaved:",
    SEQSPEC_VALIDATION_PATH,
)
print(
    "Saved:",
    REMOTE_PROBE_RESULTS_PATH,
)

if not seqspec_validation[
    "validation"
].eq("PASS").all():
    raise RuntimeError(
        "At least one seqspec failed "
        "IGVF/local/remote validation. "
        f"Review {SEQSPEC_VALIDATION_PATH} "
        f"and {REMOTE_PROBE_RESULTS_PATH}"
    )

## 11. Convert validated YAMLs to seqspec JSON attachments

Each final Portal-form YAML is converted to an object-equivalent JSON file.
The JSON file is independently checked with seqspec and attached to an IGVF
`Document` using:

```text
document_type = library structure seqspec
attachment.type = application/json
attachment.download = <seqspec>.json
```

The original validated YAML remains in `seqspec_yaml/` for review and
provenance.

In [ ]:
SEQSPEC_ATTACHMENT_MIME_TYPE = "application/json"
SEQSPEC_ATTACHMENT_DATA_PREFIX = (
    "data:application/json;base64,"
)
SEQSPEC_JSON_DIR = (
    OUTPUT_ROOT / "seqspec_json"
)
SEQSPEC_JSON_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

KNOWN_SEQSPEC_TAGS = (
    "!Assay",
    "!File",
    "!LibProtocol",
    "!LibKit",
    "!SeqProtocol",
    "!SeqKit",
    "!Read",
    "!Onlist",
    "!Region",
)


def strip_known_seqspec_tags(
    yaml_text: str,
) -> str:
    """Remove legacy seqspec YAML tags before safe loading."""
    cleaned = yaml_text
    for tag in KNOWN_SEQSPEC_TAGS:
        cleaned = re.sub(
            rf"(?m)(^|\s|:){re.escape(tag)}(?=\s|$|[:\[\]{{}},])",
            r"\1",
            cleaned,
        )
    return cleaned


def convert_seqspec_yaml_to_json(
    yaml_path: Path,
    json_path: Path,
) -> dict:
    """Write an object-equivalent JSON serialization of a seqspec YAML."""
    if not yaml_path.is_file():
        raise FileNotFoundError(yaml_path)

    yaml_text = yaml_path.read_text(
        encoding="utf-8"
    )
    spec_object = yaml.safe_load(
        strip_known_seqspec_tags(
            yaml_text
        )
    )

    if not isinstance(spec_object, dict):
        raise RuntimeError(
            f"Seqspec did not parse to a mapping: {yaml_path}"
        )

    json_path.write_text(
        json.dumps(
            spec_object,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    reloaded = json.loads(
        json_path.read_text(
            encoding="utf-8"
        )
    )
    if reloaded != spec_object:
        raise RuntimeError(
            "YAML-to-JSON object round trip changed the seqspec: "
            f"{yaml_path}"
        )

    return spec_object


def validate_portal_seqspec_json(
    *,
    json_path: Path,
    group: dict,
) -> dict:
    """Run seqspec on the JSON serialization and reject new errors."""
    if not json_path.is_file():
        raise FileNotFoundError(json_path)

    log_path = (
        LOG_DIR
        / (
            json_path.stem
            + "_check_json_attachment.log"
        )
    )

    completed = run_logged_command(
        [
            str(SEQSPEC_EXECUTABLE),
            "check",
            "--skip",
            "igvf_onlist_skip",
            str(json_path),
        ],
        log_path,
    )
    assert_check_invoked(
        completed,
        log_path,
        label=(
            "JSON attachment seqspec check"
        ),
    )

    errors = seqspec_diagnostic_lines(
        completed.stdout,
        "error",
    )
    warnings = seqspec_diagnostic_lines(
        completed.stdout,
        "warning",
    )

    expected_filenames = set(
        map(
            str,
            group[
                "portal_resource_filenames"
            ],
        )
    )

    expected_remote_errors = []
    unexpected_errors = []

    for error_line in errors:
        match = re.search(
            r"\[error\s+\d+\]\s+"
            r"(.+?)\s+does not exist$",
            error_line,
            flags=re.IGNORECASE,
        )
        if (
            match
            and match.group(1)
            in expected_filenames
        ):
            expected_remote_errors.append(
                error_line
            )
        else:
            unexpected_errors.append(
                error_line
            )

    if (
        completed.returncode != 0
        or unexpected_errors
    ):
        raise RuntimeError(
            "JSON seqspec attachment failed validation. "
            f"Review {log_path}. Unexpected errors: "
            f"{unexpected_errors}"
        )

    return {
        "json_seqspec_check_returncode": (
            completed.returncode
        ),
        "json_seqspec_expected_remote_errors": (
            " | ".join(
                expected_remote_errors
            )
        ),
        "json_seqspec_unexpected_errors": (
            " | ".join(
                unexpected_errors
            )
        ),
        "json_seqspec_warnings": (
            " | ".join(warnings)
        ),
        "json_seqspec_check_log": str(
            log_path
        ),
    }


def build_seqspec_json_attachment(
    json_path: Path,
) -> dict:
    """Create a complete application/json IGVF attachment object."""
    if (
        not json_path.is_file()
        or json_path.suffix.lower()
        != ".json"
    ):
        raise RuntimeError(
            "Seqspec attachment must be an existing .json file: "
            f"{json_path}"
        )

    json_bytes = json_path.read_bytes()
    if not json_bytes:
        raise RuntimeError(
            f"Seqspec JSON is empty: {json_path}"
        )

    return {
        "download": json_path.name,
        "href": (
            SEQSPEC_ATTACHMENT_DATA_PREFIX
            + base64.b64encode(
                json_bytes
            ).decode("ascii")
        ),
        "type": (
            SEQSPEC_ATTACHMENT_MIME_TYPE
        ),
        "md5sum": hashlib.md5(
            json_bytes
        ).hexdigest(),
        "size": len(json_bytes),
    }


def validate_seqspec_json_attachment(
    attachment: dict,
    json_path: Path,
) -> dict:
    """Validate extension, MIME metadata, and embedded bytes."""
    expected_keys = {
        "download",
        "href",
        "type",
        "md5sum",
        "size",
    }

    if not isinstance(
        attachment,
        dict,
    ):
        raise TypeError(
            "Document attachment must be a JSON object"
        )
    if set(attachment) != expected_keys:
        raise RuntimeError(
            "Unexpected attachment keys: "
            f"{sorted(attachment)}"
        )
    if "path" in attachment:
        raise RuntimeError(
            "Attachment must not contain the igvf_utils path shortcut"
        )
    if (
        attachment["download"]
        != json_path.name
        or not attachment[
            "download"
        ].endswith(".json")
    ):
        raise RuntimeError(
            "Attachment download must match the generated .json filename"
        )
    if (
        attachment["type"]
        != SEQSPEC_ATTACHMENT_MIME_TYPE
    ):
        raise RuntimeError(
            "Attachment MIME type must be application/json"
        )
    if not str(
        attachment["href"]
    ).startswith(
        SEQSPEC_ATTACHMENT_DATA_PREFIX
    ):
        raise RuntimeError(
            "Attachment href is not an application/json data URI"
        )

    encoded = str(
        attachment["href"]
    )[
        len(
            SEQSPEC_ATTACHMENT_DATA_PREFIX
        ) :
    ]
    decoded = base64.b64decode(
        encoded,
        validate=True,
    )
    expected_bytes = (
        json_path.read_bytes()
    )

    if decoded != expected_bytes:
        raise RuntimeError(
            "Attachment data URI does not reproduce the JSON bytes"
        )

    decoded_object = json.loads(
        decoded.decode("utf-8")
    )
    disk_object = json.loads(
        expected_bytes.decode(
            "utf-8"
        )
    )
    if decoded_object != disk_object:
        raise RuntimeError(
            "Decoded attachment JSON differs from the file"
        )

    observed_md5 = hashlib.md5(
        decoded
    ).hexdigest()
    if (
        attachment["md5sum"]
        != observed_md5
    ):
        raise RuntimeError(
            "Attachment MD5 does not match decoded data"
        )
    if (
        int(attachment["size"])
        != len(decoded)
    ):
        raise RuntimeError(
            "Attachment size does not match decoded data"
        )

    return {
        "attachment_download": (
            attachment["download"]
        ),
        "attachment_type": (
            attachment["type"]
        ),
        "attachment_md5sum": (
            attachment["md5sum"]
        ),
        "attachment_size": (
            attachment["size"]
        ),
        "attachment_has_path_key": (
            "path" in attachment
        ),
        "data_uri_round_trip": True,
    }


document_payloads: list[dict] = []
document_rows: list[dict] = []
attachment_preflight_rows: list[
    dict
] = []

for group in seqspec_groups.values():
    yaml_path = Path(
        group["final_yaml"]
    )

    validation_match = (
        seqspec_validation.loc[
            seqspec_validation[
                "screen"
            ].eq(
                group["screen"]
            )
            & seqspec_validation[
                "file_set"
            ].eq(
                group["file_set"]
            )
        ]
    )
    if (
        len(validation_match) != 1
        or validation_match.iloc[0][
            "validation"
        ]
        != "PASS"
    ):
        raise RuntimeError(
            "Seqspec YAML must pass validation before conversion: "
            f"{group['screen']}/{group['file_set']}"
        )

    json_path = (
        SEQSPEC_JSON_DIR
        / (
            yaml_path.stem
            + ".json"
        )
    )

    yaml_object = (
        convert_seqspec_yaml_to_json(
            yaml_path,
            json_path,
        )
    )
    json_object = json.loads(
        json_path.read_text(
            encoding="utf-8"
        )
    )
    if yaml_object != json_object:
        raise RuntimeError(
            "YAML and JSON seqspec objects differ"
        )

    json_check = (
        validate_portal_seqspec_json(
            json_path=json_path,
            group=group,
        )
    )

    attachment = (
        build_seqspec_json_attachment(
            json_path
        )
    )
    attachment_audit = (
        validate_seqspec_json_attachment(
            attachment,
            json_path,
        )
    )

    payload = {
        "aliases": [
            group["document_alias"]
        ],
        "document_type": (
            "library structure seqspec"
        ),
        "description": (
            "Seqspec for "
            + SCREEN_CONFIG[
                group["screen"]
            ]["display_name"]
            + "; file set "
            + group["file_set"]
            + ". JSON serialization of the "
            "validated seqspec."
        ),
        "attachment": attachment,
        "lab": LAB,
        "award": AWARD,
    }

    payload = json.loads(
        json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":"),
        )
    )
    validate_seqspec_json_attachment(
        payload["attachment"],
        json_path,
    )

    document_payloads.append(
        payload
    )

    json_input_path = (
        ROW_DIR
        / (
            "POST_document_"
            + safe_stem(
                group[
                    "document_alias"
                ]
            )
            + ".json"
        )
    )
    json_input_path.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    reloaded_payload = json.loads(
        json_input_path.read_text(
            encoding="utf-8"
        )
    )
    if (
        reloaded_payload
        != payload
    ):
        raise RuntimeError(
            "Document JSON input changed during disk round trip: "
            f"{json_input_path}"
        )
    validate_seqspec_json_attachment(
        reloaded_payload[
            "attachment"
        ],
        json_path,
    )

    row = {
        "screen": group["screen"],
        "file_set": (
            group["file_set"]
        ),
        "document_alias": (
            group["document_alias"]
        ),
        "source_yaml_path": str(
            yaml_path
        ),
        "attachment_json_path": str(
            json_path
        ),
        "json_input": str(
            json_input_path
        ),
        "yaml_json_object_equal": True,
        **json_check,
        **attachment_audit,
    }
    document_rows.append(row)
    attachment_preflight_rows.append(
        {
            **row,
            "status": "PASS",
        }
    )

document_inputs = pd.DataFrame(
    document_rows
)
DOCUMENT_INPUTS_PATH = (
    WORK_DIR
    / "seqspec_document_inputs.tsv"
)
write_tsv(
    document_inputs,
    DOCUMENT_INPUTS_PATH,
)

attachment_preflight = (
    pd.DataFrame(
        attachment_preflight_rows
    )
)
DOCUMENT_ATTACHMENT_PREFLIGHT_PATH = (
    WORK_DIR
    / "seqspec_document_attachment_preflight.tsv"
)
write_tsv(
    attachment_preflight,
    DOCUMENT_ATTACHMENT_PREFLIGHT_PATH,
)

# Invalidate all markers created from an older attachment representation.
(
    WORK_DIR
    / "DOCUMENT_DRY_RUN_PASSED.json"
).unlink(
    missing_ok=True
)

print(
    document_inputs.to_string(
        index=False
    )
)
print(
    "\nAttachment preflight:",
    DOCUMENT_ATTACHMENT_PREFLIGHT_PATH,
)
print(
    attachment_preflight.to_string(
        index=False
    )
)

## 12. Plan and dry-run Document submission

Existing matching Document aliases are skipped. An existing alias with a
different document type is blocking.

In [ ]:
document_plan_rows: list[dict] = []

for group in seqspec_groups.values():
    alias = group["document_alias"]

    try:
        record = get_with_retry(alias)
        lookup_error = ""
    except Exception as error:
        record = None
        lookup_error = (
            f"{type(error).__name__}: "
            f"{error}"
        )

    if lookup_error:
        action = "BLOCK_LOOKUP_ERROR"
    elif record is None:
        action = "POST_NEW"
    elif (
        str(
            record.get(
                "document_type",
                "",
            )
        )
        != "library structure seqspec"
    ):
        action = (
            "BLOCK_EXISTING_CONFLICT"
        )
    else:
        action = "SKIP_EXISTING"

    json_input = (
        document_inputs.loc[
            document_inputs[
                "document_alias"
            ].eq(alias),
            "json_input",
        ].iloc[0]
    )

    document_plan_rows.append(
        {
            "screen": group["screen"],
            "file_set": (
                group["file_set"]
            ),
            "document_alias": alias,
            "action": action,
            "json_input": json_input,
            "portal_id": (
                portal_record_id(record)
            ),
            "portal_url": (
                portal_record_url(record)
            ),
            "lookup_error": (
                lookup_error
            ),
        }
    )

document_plan = pd.DataFrame(
    document_plan_rows
)
DOCUMENT_PLAN_PATH = (
    WORK_DIR
    / "seqspec_document_submission_plan.tsv"
)
write_tsv(
    document_plan,
    DOCUMENT_PLAN_PATH,
)

print(
    document_plan.to_string(
        index=False
    )
)

if document_plan[
    "action"
].str.startswith("BLOCK_").any():
    raise RuntimeError(
        "Document plan contains a "
        f"blocking action. Review "
        f"{DOCUMENT_PLAN_PATH}"
    )

In [ ]:
RUN_DOCUMENT_DRY_RUNS = False

DOCUMENT_DRY_RUN_MARKER = (
    WORK_DIR
    / "DOCUMENT_DRY_RUN_PASSED.json"
)

if not RUN_DOCUMENT_DRY_RUNS:
    print(
        "Set RUN_DOCUMENT_DRY_RUNS=True "
        "and rerun this cell."
    )
else:
    result_rows: list[dict] = []

    for operation in (
        document_plan.to_dict(
            "records"
        )
    ):
        if (
            operation["action"]
            == "SKIP_EXISTING"
        ):
            result_rows.append(
                {
                    **operation,
                    "success": True,
                    "log_path": "",
                }
            )
            continue

        result = run_iu_register(
            profile="document",
            input_path=Path(
                operation["json_input"]
            ),
            dry_run=True,
            patch=False,
            label=(
                "seqspec_document_"
                + operation["screen"]
                + "_"
                + operation["file_set"]
            ),
        )
        result_rows.append(
            {
                **operation,
                **result,
            }
        )

    document_dry_run_results = (
        pd.DataFrame(result_rows)
    )
    DOCUMENT_DRY_RUN_RESULTS_PATH = (
        WORK_DIR
        / "document_dry_run_results.tsv"
    )
    write_tsv(
        document_dry_run_results,
        DOCUMENT_DRY_RUN_RESULTS_PATH,
    )

    if not document_dry_run_results[
        "success"
    ].all():
        raise RuntimeError(
            "At least one Document "
            "dry run failed"
        )

    marker = {
        "timestamp": time.strftime(
            "%Y-%m-%dT%H:%M:%S%z"
        ),
        "document_inputs_sha256": (
            sha256sum(
                DOCUMENT_INPUTS_PATH
            )
        ),
        "yaml_sha256": {
            row["file_set"]: (
                sha256sum(
                    Path(
                        row[
                            "final_yaml"
                        ]
                    )
                )
            )
            for row in (
                generated_yaml_rows
            )
        },
    }

    DOCUMENT_DRY_RUN_MARKER.write_text(
        json.dumps(
            marker,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    print(
        "All Document dry runs passed."
    )
    print(
        "Marker:",
        DOCUMENT_DRY_RUN_MARKER,
    )

## 13. Submit or reuse seqspec Documents

In [ ]:
REQUIRED_DOCUMENT_CONFIRMATION = (
    "POST_CRISPRI_MACS_SEQSPEC_DOCUMENTS_TO_IGVF_PROD"
)

if (
    DOCUMENT_PRODUCTION_CONFIRMATION
    != REQUIRED_DOCUMENT_CONFIRMATION
):
    print(
        "Document production submission is blocked."
    )
    print(
        "Set DOCUMENT_PRODUCTION_CONFIRMATION to:"
    )
    print(
        REQUIRED_DOCUMENT_CONFIRMATION
    )
else:
    if not DOCUMENT_DRY_RUN_MARKER.is_file():
        raise RuntimeError(
            "Document dry-run marker is missing"
        )

    marker = json.loads(
        DOCUMENT_DRY_RUN_MARKER.read_text(
            encoding="utf-8"
        )
    )

    if (
        marker["document_inputs_sha256"]
        != sha256sum(
            DOCUMENT_INPUTS_PATH
        )
    ):
        raise RuntimeError(
            "Document inputs changed after dry run"
        )

    production_rows: list[dict] = []

    for operation in document_plan.to_dict(
        "records"
    ):
        if operation["action"] == "SKIP_EXISTING":
            production_rows.append(
                {
                    **operation,
                    "success": True,
                    "log_path": "",
                }
            )
            continue

        result = run_iu_register(
            profile="document",
            input_path=Path(
                operation["json_input"]
            ),
            dry_run=False,
            patch=False,
            label=(
                "seqspec_document_"
                + operation["screen"]
                + "_"
                + operation["file_set"]
            ),
        )
        production_rows.append(
            {
                **operation,
                **result,
            }
        )

    document_production_results = pd.DataFrame(
        production_rows
    )
    DOCUMENT_PRODUCTION_RESULTS_PATH = (
        WORK_DIR
        / "document_production_results.tsv"
    )
    write_tsv(
        document_production_results,
        DOCUMENT_PRODUCTION_RESULTS_PATH,
    )

    verified_rows: list[dict] = []

    for group in seqspec_groups.values():
        record = get_with_retry(
            group["document_alias"],
            attempts=10,
        )

        if record is None:
            raise RuntimeError(
                "Document not found after submission: "
                f"{group['document_alias']}"
            )

        if (
            str(record.get("document_type", ""))
            != "library structure seqspec"
        ):
            raise RuntimeError(
                "Unexpected document_type for "
                f"{group['document_alias']}"
            )

        portal_attachment = (
            record.get("attachment")
            or {}
        )
        expected_json_name = (
            Path(group["final_yaml"]).stem
            + ".json"
        )

        if (
            portal_attachment.get("type")
            != "application/json"
        ):
            raise RuntimeError(
                "Unexpected Portal attachment MIME type for "
                f"{group['document_alias']}: "
                f"{portal_attachment.get('type')!r}"
            )
        if (
            portal_attachment.get("download")
            != expected_json_name
        ):
            raise RuntimeError(
                "Unexpected Portal seqspec JSON attachment filename for "
                f"{group['document_alias']}"
            )

        verified_rows.append(
            {
                "screen": group["screen"],
                "file_set": group["file_set"],
                "document_alias": group["document_alias"],
                "document_id": portal_record_id(record),
                "document_url": portal_record_url(record),
                "attachment_download": portal_attachment.get(
                    "download",
                    "",
                ),
                "attachment_type": portal_attachment.get(
                    "type",
                    "",
                ),
                "verification": "PASS",
            }
        )

    document_records = pd.DataFrame(
        verified_rows
    )
    DOCUMENT_RECORDS_PATH = (
        WORK_DIR
        / "seqspec_document_portal_records.csv"
    )
    document_records.to_csv(
        DOCUMENT_RECORDS_PATH,
        index=False,
        lineterminator="\n",
    )

    print(
        document_records.to_string(
            index=False
        )
    )

## 14. Build SequenceFile PATCH operations

`seqspec_document` is singular. Each SequenceFile is assigned the Document
for its own screen/file-set group. Already-correct links are skipped.

In [ ]:
document_records_by_group: dict[
    tuple[str, str],
    dict,
] = {}

for group in seqspec_groups.values():
    record = get_with_retry(
        group["document_alias"],
        attempts=5,
    )
    if record is None:
        raise RuntimeError(
            "Required seqspec Document "
            "does not exist: "
            f"{group['document_alias']}"
        )

    document_records_by_group[
        (
            group["screen"],
            group["file_set"],
        )
    ] = record

patch_plan_rows: list[dict] = []

for row in canonical_sequence_files.to_dict(
    "records"
):
    key = (
        row["screen"],
        row["file_set"],
    )
    document_record = (
        document_records_by_group[key]
    )
    document_id = portal_record_id(
        document_record
    )

    sequence_record = get_with_retry(
        row["accession"],
        attempts=5,
    )
    if sequence_record is None:
        action = "BLOCK_FILE_NOT_FOUND"
        existing_document = ""
    else:
        existing_document = (
            sequence_record.get(
                "seqspec_document",
                "",
            )
        )

        if links_equal(
            existing_document,
            document_id,
        ):
            action = (
                "SKIP_ALREADY_LINKED"
            )
        elif existing_document:
            action = (
                "BLOCK_EXISTING_DIFFERENT_"
                "SEQSPEC"
            )
        else:
            action = "PATCH_FILE"

    patch_plan_rows.append(
        {
            "screen": row["screen"],
            "file_set": row["file_set"],
            "accession": (
                row["accession"]
            ),
            "path": row["path"],
            "document_alias": (
                seqspec_groups[key][
                    "document_alias"
                ]
            ),
            "document_id": document_id,
            "existing_seqspec_document": (
                ",".join(
                    sorted(
                        direct_link_tokens(
                            existing_document
                        )
                    )
                )
            ),
            "action": action,
            "portal_url": (
                portal_record_url(
                    sequence_record
                )
            ),
        }
    )

patch_plan = pd.DataFrame(
    patch_plan_rows
)
PATCH_PLAN_PATH = (
    WORK_DIR
    / "sequencefile_seqspec_patch_plan.tsv"
)
write_tsv(
    patch_plan,
    PATCH_PLAN_PATH,
)

print(
    patch_plan.to_string(
        index=False
    )
)

if patch_plan[
    "action"
].str.startswith("BLOCK_").any():
    raise RuntimeError(
        "SequenceFile patch plan "
        f"is blocked. Review "
        f"{PATCH_PLAN_PATH}"
    )

patch_operation_rows: list[dict] = []

for plan in patch_plan.to_dict(
    "records"
):
    if plan["action"] == "PATCH_FILE":
        tsv_path = (
            ROW_DIR
            / (
                "PATCH_sequence_file_"
                + plan["accession"]
                + ".tsv"
            )
        )

        pd.DataFrame(
            [
                {
                    "record_id": (
                        plan["accession"]
                    ),
                    "seqspec_document": (
                        plan["document_id"]
                    ),
                }
            ]
        ).to_csv(
            tsv_path,
            sep="\t",
            index=False,
            lineterminator="\n",
        )
    else:
        tsv_path = None

    patch_operation_rows.append(
        {
            "screen": plan["screen"],
            "file_set": (
                plan["file_set"]
            ),
            "accession": (
                plan["accession"]
            ),
            "action": plan["action"],
            "input_tsv": (
                str(tsv_path)
                if tsv_path
                else ""
            ),
        }
    )

patch_operations = pd.DataFrame(
    patch_operation_rows
)
PATCH_OPERATIONS_PATH = (
    WORK_DIR
    / "sequencefile_seqspec_patch_operations.tsv"
)
write_tsv(
    patch_operations,
    PATCH_OPERATIONS_PATH,
)

# Explicit path preflight.
path_rows: list[dict] = []
for operation in (
    patch_operations.to_dict(
        "records"
    )
):
    input_text = str(
        operation["input_tsv"]
    ).strip()
    required = (
        operation["action"]
        == "PATCH_FILE"
    )
    exists = (
        Path(input_text).is_file()
        if input_text
        else False
    )

    path_rows.append(
        {
            **operation,
            "requires_file": required,
            "input_exists": exists,
            "status": (
                "PASS"
                if (
                    not required
                    or exists
                )
                else "FAIL"
            ),
        }
    )

patch_path_preflight = pd.DataFrame(
    path_rows
)
PATCH_PATH_PREFLIGHT_PATH = (
    WORK_DIR
    / "sequencefile_patch_path_preflight.tsv"
)
write_tsv(
    patch_path_preflight,
    PATCH_PATH_PREFLIGHT_PATH,
)

print(
    "\nPatch operations:"
)
print(
    patch_path_preflight.to_string(
        index=False
    )
)

if not patch_path_preflight[
    "status"
].eq("PASS").all():
    raise RuntimeError(
        "At least one SequenceFile "
        "PATCH TSV is missing"
    )

## 15. Dry-run SequenceFile PATCHes

In [ ]:
RUN_SEQUENCE_PATCH_DRY_RUNS = False

SEQUENCE_PATCH_DRY_RUN_MARKER = (
    WORK_DIR
    / "SEQUENCE_PATCH_DRY_RUN_PASSED.json"
)

if not RUN_SEQUENCE_PATCH_DRY_RUNS:
    print(
        "Set RUN_SEQUENCE_PATCH_DRY_RUNS=True "
        "and rerun this cell."
    )
else:
    result_rows: list[dict] = []

    for operation in (
        patch_operations.to_dict(
            "records"
        )
    ):
        if (
            operation["action"]
            == "SKIP_ALREADY_LINKED"
        ):
            result_rows.append(
                {
                    **operation,
                    "success": True,
                    "log_path": "",
                }
            )
            continue

        input_text = str(
            operation["input_tsv"]
        ).strip()
        if not input_text:
            raise RuntimeError(
                "PATCH operation has no "
                f"input TSV: {operation}"
            )
        input_path = Path(input_text)
        if not input_path.is_file():
            raise FileNotFoundError(
                input_path
            )

        result = run_iu_register(
            profile="sequence_file",
            input_path=input_path,
            dry_run=True,
            patch=True,
            label=(
                "sequencefile_seqspec_"
                + operation["accession"]
            ),
        )
        result_rows.append(
            {
                **operation,
                **result,
            }
        )

    patch_dry_run_results = (
        pd.DataFrame(result_rows)
    )
    PATCH_DRY_RUN_RESULTS_PATH = (
        WORK_DIR
        / "sequencefile_patch_dry_run_results.tsv"
    )
    write_tsv(
        patch_dry_run_results,
        PATCH_DRY_RUN_RESULTS_PATH,
    )

    if not patch_dry_run_results[
        "success"
    ].all():
        raise RuntimeError(
            "At least one SequenceFile "
            "PATCH dry run failed"
        )

    marker = {
        "timestamp": time.strftime(
            "%Y-%m-%dT%H:%M:%S%z"
        ),
        "patch_plan_sha256": (
            sha256sum(
                PATCH_PLAN_PATH
            )
        ),
        "operation_sha256": {
            operation["accession"]: (
                sha256sum(
                    Path(
                        operation[
                            "input_tsv"
                        ]
                    )
                )
                if operation["input_tsv"]
                else ""
            )
            for operation in (
                patch_operation_rows
            )
        },
    }

    SEQUENCE_PATCH_DRY_RUN_MARKER.write_text(
        json.dumps(
            marker,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    print(
        "All SequenceFile PATCH "
        "dry runs passed."
    )
    print(
        "Marker:",
        SEQUENCE_PATCH_DRY_RUN_MARKER,
    )

## 16. Apply SequenceFile PATCHes

In [ ]:
REQUIRED_SEQUENCE_PATCH_CONFIRMATION = (
    "ATTACH_CRISPRI_MACS_SEQSPEC_DOCUMENTS_TO_SEQUENCEFILES"
)

if (
    SEQUENCE_PATCH_PRODUCTION_CONFIRMATION
    != REQUIRED_SEQUENCE_PATCH_CONFIRMATION
):
    print(
        "Production SequenceFile PATCH "
        "is blocked."
    )
    print(
        "Set SEQUENCE_PATCH_PRODUCTION_CONFIRMATION "
        "to:"
    )
    print(
        REQUIRED_SEQUENCE_PATCH_CONFIRMATION
    )
else:
    if not (
        SEQUENCE_PATCH_DRY_RUN_MARKER
        .is_file()
    ):
        raise RuntimeError(
            "SequenceFile PATCH "
            "dry-run marker is missing"
        )

    marker = json.loads(
        SEQUENCE_PATCH_DRY_RUN_MARKER.read_text(
            encoding="utf-8"
        )
    )

    if (
        marker["patch_plan_sha256"]
        != sha256sum(
            PATCH_PLAN_PATH
        )
    ):
        raise RuntimeError(
            "SequenceFile patch plan "
            "changed after dry run"
        )

    production_rows: list[dict] = []

    for operation in (
        patch_operations.to_dict(
            "records"
        )
    ):
        if (
            operation["action"]
            == "SKIP_ALREADY_LINKED"
        ):
            production_rows.append(
                {
                    **operation,
                    "success": True,
                    "log_path": "",
                }
            )
            continue

        input_path = Path(
            operation["input_tsv"]
        )

        if (
            marker[
                "operation_sha256"
            ][operation["accession"]]
            != sha256sum(input_path)
        ):
            raise RuntimeError(
                "PATCH TSV changed after "
                f"dry run: {input_path}"
            )

        result = run_iu_register(
            profile="sequence_file",
            input_path=input_path,
            dry_run=False,
            patch=True,
            label=(
                "sequencefile_seqspec_"
                + operation["accession"]
            ),
        )
        production_rows.append(
            {
                **operation,
                **result,
            }
        )

    production_results = pd.DataFrame(
        production_rows
    )
    PATCH_PRODUCTION_RESULTS_PATH = (
        WORK_DIR
        / "sequencefile_patch_production_results.tsv"
    )
    write_tsv(
        production_results,
        PATCH_PRODUCTION_RESULTS_PATH,
    )

    print(
        production_results.to_string(
            index=False
        )
    )

## 17. Final read-only reconciliation

In [ ]:
reconciliation_rows: list[dict] = []

for row in canonical_sequence_files.to_dict(
    "records"
):
    key = (
        row["screen"],
        row["file_set"],
    )
    document_alias = (
        seqspec_groups[key][
            "document_alias"
        ]
    )

    sequence_record = get_with_retry(
        row["accession"],
        attempts=5,
    )
    document_record = get_with_retry(
        document_alias,
        attempts=5,
    )

    document_id = portal_record_id(
        document_record
    )
    existing_document = (
        (sequence_record or {}).get(
            "seqspec_document",
            "",
        )
    )

    linked = bool(
        sequence_record
        and document_record
        and links_equal(
            existing_document,
            document_id,
        )
    )

    reconciliation_rows.append(
        {
            "screen": row["screen"],
            "file_set": (
                row["file_set"]
            ),
            "sequence_file_accession": (
                row["accession"]
            ),
            "sequence_file_url": (
                portal_record_url(
                    sequence_record
                )
            ),
            "document_alias": (
                document_alias
            ),
            "document_id": document_id,
            "document_url": (
                portal_record_url(
                    document_record
                )
            ),
            "portal_seqspec_document": (
                ",".join(
                    sorted(
                        direct_link_tokens(
                            existing_document
                        )
                    )
                )
            ),
            "verification": (
                "PASS"
                if linked
                else "FAIL"
            ),
        }
    )

reconciliation = pd.DataFrame(
    reconciliation_rows
)
FINAL_RECONCILIATION_PATH = (
    OUTPUT_ROOT
    / "sequencefile_seqspec_reconciliation.csv"
)
reconciliation.to_csv(
    FINAL_RECONCILIATION_PATH,
    index=False,
    lineterminator="\n",
)

print(
    reconciliation.to_string(
        index=False
    )
)
print(
    "\nSaved:",
    FINAL_RECONCILIATION_PATH,
)

passed = int(
    reconciliation[
        "verification"
    ].eq("PASS").sum()
)
total = len(reconciliation)

print(
    f"\nLinked SequenceFiles: "
    f"{passed}/{total}"
)

if total and passed == total:
    marker = (
        OUTPUT_ROOT
        / (
            "SEQSPEC_DOCUMENTS_LINKED_"
            f"{total}_OF_{total}.txt"
        )
    )
    marker.write_text(
        "All reconciled SequenceFiles "
        "link to their expected seqspec "
        "Documents.\n"
        f"Results: "
        f"{FINAL_RECONCILIATION_PATH}\n",
        encoding="utf-8",
    )
    print(
        "Completion marker:",
        marker,
    )

## 18. Output summary

In [ ]:
output_candidates = [
    (
        "guide library inventory",
        GUIDE_LIBRARY_INVENTORY_PATH,
    ),
    (
        "local FASTQ discovery",
        LOCAL_FASTQ_DISCOVERY_PATH,
    ),
    (
        "local-Portal crosswalk",
        SEQUENCE_CROSSWALK_PATH,
    ),
    (
        "seqspec groups",
        GROUP_SUMMARY_PATH,
    ),
    (
        "guide-layout inference",
        LAYOUT_INFERENCE_PATH,
    ),
    (
        "guide-layout scores",
        LAYOUT_SCORES_PATH,
    ),
    (
        "generated YAML inventory",
        GENERATED_YAML_INVENTORY_PATH,
    ),
    (
        "seqspec validation",
        SEQSPEC_VALIDATION_PATH,
    ),
    (
        "Document inputs",
        DOCUMENT_INPUTS_PATH,
    ),
    (
        "Document attachment preflight",
        WORK_DIR / "seqspec_document_attachment_preflight.tsv",
    ),
    (
        "Document plan",
        DOCUMENT_PLAN_PATH,
    ),
    (
        "SequenceFile patch plan",
        PATCH_PLAN_PATH,
    ),
    (
        "final reconciliation",
        OUTPUT_ROOT
        / "sequencefile_seqspec_reconciliation.csv",
    ),
]

output_summary = pd.DataFrame(
    [
        {
            "output": label,
            "path": str(path),
            "exists": path.is_file(),
            "size_bytes": (
                path.stat().st_size
                if path.is_file()
                else 0
            ),
        }
        for label, path in (
            output_candidates
        )
    ]
)

print(
    output_summary.to_string(
        index=False
    )
)
print("\nYAML directory:", YAML_DIR)
print("Seqspec JSON directory:", SEQSPEC_JSON_DIR)
print("Logs:", LOG_DIR)